
# 14. Crop–Weather Data Integration

## 1. Objective

The objective of this notebook is to integrate the preprocessed Area, Production and Yield (APY) dataset with the annual district-level weather dataset for BharatCrop Intelligence.

This integration will establish a unified analytical dataset for subsequent feature engineering, crop-yield prediction, crop recommendation, and agricultural risk analysis.

## 2. Integration Strategy

The integration process will follow these steps:

1. Identify and load the existing preprocessed APY and weather datasets.
2. Inspect their schemas, data types, geographic identifiers, and temporal coverage.
3. Standardize state and district names without losing the original values.
4. examine the APY agricultural-year convention and the weather calendar-year convention.
5. Define and document a defensible temporal alignment strategy.
6. Integrate the datasets using validated geographic and temporal keys.
7. Evaluate unmatched records, duplicate keys, missing weather values, and row-count changes.
8. Export the integrated dataset only after the validation checks pass.

## 3. Key Considerations

- APY agricultural years and weather calendar years must not be assumed to be equivalent.
- State and district names may differ in spelling, formatting, or administrative naming conventions.
- Aggregate crop categories must be distinguished from individual crops to avoid misleading analyses.
- Annual weather summaries are contextual indicators; they do not represent crop-stage-specific weather exposure.
- Missing or unmatched records must be measured and reported rather than silently discarded.
- The original source datasets must remain unchanged.

## 4. Expected Outputs

- A documented integration strategy.
- A geographic and temporal matching assessment.
- Integration quality-control results.
- A validated crop–weather dataset for downstream analysis.

No predictive model will be trained in this notebook.


## 5. Project Paths and Dataset Discovery

Before loading or joining the datasets, we must locate the project root and inspect the available processed files.

This step is read-only. It will help us identify the exact APY and weather datasets produced by the preceding preprocessing notebooks, avoiding assumptions about filenames or directory structures.

**Validation goals:**
- Locate the project root.
- Identify candidate APY and weather files.
- Confirm the files exist before proceeding to schema inspection.



In [1]:
from pathlib import Path
import os

# Locate the project root by searching the current directory and its parents.
current_path = Path.cwd().resolve()
project_root = None

for path in [current_path, *current_path.parents]:
    if (path / "data" / "processed").is_dir():
        project_root = path
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Run this notebook from inside BharatCrop-Intelligence "
        "or update the project path manually."
    )

processed_dir = project_root / "data" / "processed"

print(f"Project root: {project_root}")
print(f"Processed data directory: {processed_dir}")
print("\nAvailable processed datasets:\n")

dataset_files = sorted(
    path for path in processed_dir.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".csv", ".xlsx", ".xls", ".parquet"}
)

if not dataset_files:
    print("No CSV, Excel, or Parquet files found.")
else:
    for file_path in dataset_files:
        relative_path = file_path.relative_to(project_root)
        filename = file_path.name.lower()
        tags = []

        if any(term in filename for term in ["apy", "crop", "yield", "production"]):
            tags.append("Possible APY/crop dataset")

        if any(term in filename for term in ["weather", "climate", "precip", "temperature"]):
            tags.append("Possible weather dataset")

        tag_text = f"  [{'; '.join(tags)}]" if tags else ""
        print(f"{relative_path}{tag_text}")

print(f"\nTotal processed files found: {len(dataset_files)}")

Project root: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence
Processed data directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed

Available processed datasets:

data\processed\.ipynb_checkpoints\historical_food_grains_apy_2013-14_to_2024-25-checkpoint.csv  [Possible APY/crop dataset]
data\processed\historical_food_grains_apy_2013-14_to_2024-25.csv  [Possible APY/crop dataset]
data\processed\historical_food_grains_apy_cleaned_flagged.csv  [Possible APY/crop dataset]
data\processed\historical_food_grains_apy_long_2013-14_to_2024-25.csv  [Possible APY/crop dataset]
data\processed\weather_pilot_2013_2024\nasa_power_weather_pilot_2013_2024.csv  [Possible weather dataset]
data\processed\weather_regional\district_weather_annual_2013_2024.csv  [Possible weather dataset]
data\processed\weather_regional\district_weather_annual_cleaned_flagged_2013_2024.csv  [Possible weather dataset]
data\processed\weather_regional\district_weather_annual_preprocessed_2013_2

In [2]:
import pandas as pd
from pathlib import Path

# Reuse project_root discovered in Cell 3.
apy_path = (
    project_root
    / "data"
    / "processed"
    / "historical_food_grains_apy_long_2013-14_to_2024-25.csv"
)

weather_path = (
    project_root
    / "data"
    / "processed"
    / "weather_regional"
    / "district_weather_annual_preprocessed_2013_2024.csv"
)

# Verify both files exist before loading.
for file_path in [apy_path, weather_path]:
    if not file_path.is_file():
        raise FileNotFoundError(f"Dataset not found: {file_path}")

# Load the datasets.
apy_df = pd.read_csv(apy_path, low_memory=False)
weather_df = pd.read_csv(weather_path, low_memory=False)

def inspect_dataset(df, name):
    print("=" * 80)
    print(f"{name}")
    print("=" * 80)

    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

    print("\nColumn names and data types:")
    print(df.dtypes.to_string())

    print("\nFirst 5 records:")
    display(df.head())

    print("\nMissing values (top 15 columns):")
    missing = df.isna().sum().sort_values(ascending=False)
    print(missing.head(15).to_string())

    print(f"\nExact duplicate rows: {df.duplicated().sum():,}")

    print("\nPotential geographic and temporal columns:")
    candidates = [
        col for col in df.columns
        if any(term in col.lower() for term in [
            "state", "district", "year", "season", "date"
        ])
    ]
    print(candidates if candidates else "No obvious candidates found.")

inspect_dataset(apy_df, "APY DATASET")
inspect_dataset(weather_df, "WEATHER DATASET")

APY DATASET
Shape: 376,272 rows × 13 columns

Column names and data types:
State                                             object
District                                          object
Crop                                              object
Season                                            object
Crop_Category                                     object
Year                                              object
Area                                             float64
Production                                       float64
Yield                                            float64
QC_AreaZero-ProductionPositive                      bool
QC_AreaPositive-ProductionZero                      bool
QC_NegativeMeasurements                             bool
QC_PositiveYieldWithoutPositiveAreaProduction       bool

First 5 records:


,State,District,Crop,Season,Crop_Category,Year,Area,Production,Yield,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_NegativeMeasurements,QC_PositiveYieldWithoutPositiveAreaProduction
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,Grouping — Verify Source Definition,2013-14,0.0,0.0,2377.0,False,False,False,True
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,Grouping — Verify Source Definition,2013-14,0.0,0.0,526.0,False,False,False,True
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,Grouping — Verify Source Definition,2013-14,0.0,0.0,923.0,False,False,False,True
3,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,Individual Crop Candidate,2013-14,0.0,0.0,526.0,False,False,False,True
4,Andaman And Nicobar Islands,Nicobars,Maize,Total,Individual Crop Candidate,2013-14,0.0,0.0,526.0,False,False,False,True



Missing values (top 15 columns):
Yield                                            95033
Production                                       95030
Area                                             94183
District                                             0
State                                                0
Crop_Category                                        0
Season                                               0
Crop                                                 0
Year                                                 0
QC_AreaZero-ProductionPositive                       0
QC_AreaPositive-ProductionZero                       0
QC_NegativeMeasurements                              0
QC_PositiveYieldWithoutPositiveAreaProduction        0

Exact duplicate rows: 0

Potential geographic and temporal columns:
['State', 'District', 'Season', 'Year']
WEATHER DATASET
Shape: 7,680 rows × 17 columns

Column names and data types:
apy_geography_id                      object
calendar_year       

,apy_geography_id,calendar_year,tmin_mean_c,tmin_min_c,tmax_mean_c,tmax_max_c,precipitation_total_mm,precipitation_mean_mm_day,precipitation_max_mm_day,tmin_valid_days,tmax_valid_days,precipitation_valid_days,QC_Extreme_Annual_Min_Temperature,QC_Extreme_Annual_Max_Temperature,QC_High_Annual_Precipitation,QC_Extreme_Daily_Precipitation,QC_Long_District_Grid_Distance
0,APY-GEO-0001,2013,27.472904,25.78,28.279671,30.11,2456.73,6.730767,29.00,365,365,365,False,False,False,False,False
1,APY-GEO-0001,2014,27.517479,25.74,28.430301,30.75,1972.96,5.405370,51.98,365,365,365,False,False,False,False,False
2,APY-GEO-0001,2015,27.664959,26.03,28.533370,30.15,1767.49,4.842438,36.97,365,365,365,False,False,False,False,False
3,APY-GEO-0001,2016,27.918333,25.43,28.810492,31.98,2187.42,5.976557,52.15,366,366,366,False,False,False,False,False
4,APY-GEO-0001,2017,27.570219,25.54,28.360055,29.62,2155.98,5.906795,48.42,365,365,365,False,False,False,False,False



Missing values (top 15 columns):
apy_geography_id                     0
calendar_year                        0
tmin_mean_c                          0
tmin_min_c                           0
tmax_mean_c                          0
tmax_max_c                           0
precipitation_total_mm               0
precipitation_mean_mm_day            0
precipitation_max_mm_day             0
tmin_valid_days                      0
tmax_valid_days                      0
precipitation_valid_days             0
QC_Extreme_Annual_Min_Temperature    0
QC_Extreme_Annual_Max_Temperature    0
QC_High_Annual_Precipitation         0

Exact duplicate rows: 0

Potential geographic and temporal columns:
['calendar_year', 'QC_Long_District_Grid_Distance']


In [3]:
# 1. Inspect APY agricultural-year values.
print("=" * 70)
print("APY YEAR CONVENTION")
print("=" * 70)

apy_years = sorted(apy_df["Year"].dropna().astype(str).unique())

print("Number of unique APY years:", len(apy_years))
print("First 15 APY years:", apy_years[:15])
print("Last 10 APY years:", apy_years[-10:])

# 2. Inspect weather calendar-year coverage and geography IDs.
print("\n" + "=" * 70)
print("WEATHER TEMPORAL AND GEOGRAPHIC COVERAGE")
print("=" * 70)

print(
    "Calendar-year range:",
    weather_df["calendar_year"].min(),
    "to",
    weather_df["calendar_year"].max()
)
print("Unique weather geography IDs:", weather_df["apy_geography_id"].nunique())
print("Unique weather years:", weather_df["calendar_year"].nunique())

print("\nSample geography IDs:")
print(weather_df["apy_geography_id"].drop_duplicates().head(10).to_list())

# 3. Check whether each weather geography ID has one record per year.
weather_key = ["apy_geography_id", "calendar_year"]

print(
    "\nDuplicate weather geography-year keys:",
    weather_df.duplicated(subset=weather_key).sum()
)

# 4. Search project files for possible geography mapping sources.
print("\n" + "=" * 70)
print("POSSIBLE GEOGRAPHY MAPPING FILES")
print("=" * 70)

mapping_candidates = []

for file_path in project_root.rglob("*"):
    if not file_path.is_file():
        continue

    # Ignore notebook checkpoints and common generated folders.
    if any(part.lower() in {".ipynb_checkpoints", ".git", "__pycache__"}
           for part in file_path.parts):
        continue

    name = file_path.name.lower()

    if (
        any(term in name for term in ["geography", "geographic", "district_mapping",
                                      "district_map", "geo_mapping", "crosswalk"])
        and file_path.suffix.lower() in {".csv", ".xlsx", ".xls", ".parquet"}
    ):
        mapping_candidates.append(file_path)

if mapping_candidates:
    for file_path in mapping_candidates:
        print(file_path.relative_to(project_root))
else:
    print("No obvious geography mapping file found.")

# 5. Check whether APY district-state pairs are consistent.
print("\n" + "=" * 70)
print("APY GEOGRAPHIC IDENTIFIER CHECK")
print("=" * 70)

state_counts = apy_df.groupby("District")["State"].nunique()
ambiguous_districts = state_counts[state_counts > 1]

print("Unique states:", apy_df["State"].nunique())
print("Unique districts:", apy_df["District"].nunique())
print("District names associated with multiple states:", len(ambiguous_districts))

if not ambiguous_districts.empty:
    print("\nExamples:")
    print(ambiguous_districts.head(10).to_string())

APY YEAR CONVENTION
Number of unique APY years: 12
First 15 APY years: ['2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
Last 10 APY years: ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']

WEATHER TEMPORAL AND GEOGRAPHIC COVERAGE
Calendar-year range: 2013 to 2024
Unique weather geography IDs: 640
Unique weather years: 12

Sample geography IDs:
['APY-GEO-0001', 'APY-GEO-0002', 'APY-GEO-0003', 'APY-GEO-0005', 'APY-GEO-0006', 'APY-GEO-0007', 'APY-GEO-0008', 'APY-GEO-0009', 'APY-GEO-0010', 'APY-GEO-0011']

Duplicate weather geography-year keys: 0

POSSIBLE GEOGRAPHY MAPPING FILES
metadata\apy_district_coordinate_crosswalk.csv
metadata\apy_district_geographic_reference.csv
notebooks\reports\data_audit\apy_exact_name_crosswalk_review.csv

APY GEOGRAPHIC IDENTIFIER CHECK
Unique states: 34
Unique districts: 748
District names associated with multiple

In [4]:
from pathlib import Path
import pandas as pd

metadata_dir = project_root / "metadata"

reference_path = metadata_dir / "apy_district_geographic_reference.csv"
crosswalk_path = metadata_dir / "apy_district_coordinate_crosswalk.csv"

for path in [reference_path, crosswalk_path]:
    print("\n" + "=" * 80)
    print(f"FILE: {path.relative_to(project_root)}")
    print("=" * 80)

    if not path.is_file():
        print("File not found.")
        continue

    df = pd.read_csv(path, low_memory=False)

    print(f"Shape: {df.shape}")
    print("\nColumns and data types:")
    print(df.dtypes.to_string())

    print("\nFirst 5 rows:")
    display(df.head())

    print("\nPotential ID/geography columns:")
    print([
        col for col in df.columns
        if any(term in col.lower() for term in [
            "apy", "geo", "state", "district", "latitude",
            "longitude", "year", "id", "name"
        ])
    ])

# Inspect the reviewed exact-name crosswalk, if present.
review_path = (
    project_root
    / "notebooks"
    / "reports"
    / "data_audit"
    / "apy_exact_name_crosswalk_review.csv"
)

if review_path.is_file():
    print("\n" + "=" * 80)
    print(f"FILE: {review_path.relative_to(project_root)}")
    print("=" * 80)

    review_df = pd.read_csv(review_path, low_memory=False)

    print(f"Shape: {review_df.shape}")
    print("\nColumns and data types:")
    print(review_df.dtypes.to_string())

    print("\nFirst 10 rows:")
    display(review_df.head(10))
else:
    print(f"\nReview file not found: {review_path}")


FILE: metadata\apy_district_geographic_reference.csv
Shape: (751, 8)

Columns and data types:
apy_geography_id          object
State                     object
District                  object
boundary_match_status     object
matched_boundary_name    float64
latitude                 float64
longitude                float64
match_notes              float64

First 5 rows:


,apy_geography_id,State,District,boundary_match_status,matched_boundary_name,latitude,longitude,match_notes
0,APY-GEO-0001,Andaman And Nicobar Islands,Nicobars,NOT_MATCHED,NaN,NaN,NaN,NaN
1,APY-GEO-0002,Andaman And Nicobar Islands,North And Middle Andaman,NOT_MATCHED,NaN,NaN,NaN,NaN
2,APY-GEO-0003,Andaman And Nicobar Islands,South Andamans,NOT_MATCHED,NaN,NaN,NaN,NaN
3,APY-GEO-0004,Andhra Pradesh,Alluri Sitharama Raju,NOT_MATCHED,NaN,NaN,NaN,NaN
4,APY-GEO-0005,Andhra Pradesh,Anakapalli,NOT_MATCHED,NaN,NaN,NaN,NaN



Potential ID/geography columns:
['apy_geography_id', 'State', 'District', 'matched_boundary_name', 'latitude', 'longitude']

FILE: metadata\apy_district_coordinate_crosswalk.csv
Shape: (751, 12)

Columns and data types:
apy_geography_id          object
State                     object
District                  object
latitude                 float64
longitude                float64
coordinate_source         object
is_synthetic                bool
geography_confidence      object
matched_boundary_name     object
state_lgd                float64
district_lgd             float64
match_notes               object

First 5 rows:


,apy_geography_id,State,District,latitude,longitude,coordinate_source,is_synthetic,geography_confidence,matched_boundary_name,state_lgd,district_lgd,match_notes
0,APY-GEO-0001,Andaman And Nicobar Islands,Nicobars,6.998715,93.823650,BOUNDARY_REPRESENTATIVE_POINT,False,PROVISIONAL_BOUNDARY_MATCH,NICOBARS,35.0,638.0,Representative point from boundary polygon. Hi...
1,APY-GEO-0002,Andaman And Nicobar Islands,North And Middle Andaman,13.228353,92.952877,BOUNDARY_REPRESENTATIVE_POINT,False,PROVISIONAL_BOUNDARY_MATCH,NORTH AND MIDDLE ANDAMAN,35.0,639.0,Representative point from boundary polygon. Hi...
2,APY-GEO-0003,Andaman And Nicobar Islands,South Andamans,10.706412,92.489082,BOUNDARY_REPRESENTATIVE_POINT,False,PROVISIONAL_BOUNDARY_MATCH,SOUTH ANDAMANS,35.0,640.0,Representative point from boundary polygon. Hi...
3,APY-GEO-0004,Andhra Pradesh,Alluri Sitharama Raju,16.500000,80.600000,SYNTHETIC_STATE_ANCHOR_OFFSET,True,LOW_SYNTHETIC_PLACEHOLDER,NaN,NaN,NaN,Synthetic placeholder near state anchor; not a...
4,APY-GEO-0005,Andhra Pradesh,Anakapalli,17.671641,82.774670,BOUNDARY_REPRESENTATIVE_POINT,False,PROVISIONAL_BOUNDARY_MATCH,Anakapalli,28.0,685.0,Representative point from boundary polygon. Hi...



Potential ID/geography columns:
['apy_geography_id', 'State', 'District', 'latitude', 'longitude', 'geography_confidence', 'matched_boundary_name', 'state_lgd', 'district_lgd']

FILE: notebooks\reports\data_audit\apy_exact_name_crosswalk_review.csv
Shape: (34, 11)

Columns and data types:
apy_geography_id                object
apy_state                       object
apy_district                    object
boundary_district               object
boundary_dist_lgd                int64
APY_normalized_name             object
candidate_1_normalized          object
candidate_status                object
match_decision                  object
geographic_vintage_verified       bool
review_notes                   float64

First 10 rows:


,apy_geography_id,apy_state,apy_district,boundary_district,boundary_dist_lgd,APY_normalized_name,candidate_1_normalized,candidate_status,match_decision,geographic_vintage_verified,review_notes
0,APY-GEO-0030,Arunachal Pradesh,Anjaw,>NJ>W,260,ANJAW,ANJAW,EXACT_NORMALIZED_CANDIDATE_REQUIRES_REVIEW,PENDING,False,NaN
1,APY-GEO-0031,Arunachal Pradesh,Changlang,CH>NGL>NG,253,CHANGLANG,CHANGLANG,EXACT_NORMALIZED_CANDIDATE_REQUIRES_REVIEW,PENDING,False,NaN
2,APY-GEO-0032,Arunachal Pradesh,Dibang Valley,DIB>NG VALLEY,257,DIBANG VALLEY,DIBANG VALLEY,EXACT_NORMALIZED_CANDIDATE_REQUIRES_REVIEW,PENDING,False,NaN
3,APY-GEO-0041,Arunachal Pradesh,Lower Dibang Valley,LOWER DIB>NG VALLEY,258,LOWER DIBANG VALLEY,LOWER DIBANG VALLEY,EXACT_NORMALIZED_CANDIDATE_REQUIRES_REVIEW,PENDING,False,NaN
4,APY-GEO-0050,Arunachal Pradesh,Tirap,TIR>P,254,TIRAP,TIRAP,EXACT_NORMALIZED_CANDIDATE_REQUIRES_REVIEW,PENDING,False,NaN
5,APY-GEO-0139,Chhattisgarh,Janjgir-Champa,JANJGIR - CHAMPA,405,JANJGIR CHAMPA,JANJGIR CHAMPA,EXACT_NORMALIZED_CANDIDATE_REQUIRES_REVIEW,PENDING,False,NaN
6,APY-GEO-0217,Himachal Pradesh,Bilaspur,BIL>SPUR,30,BILASPUR,BILASPUR,EXACT_NORMALIZED_CANDIDATE_REQUIRES_REVIEW,PENDING,False,NaN
7,APY-GEO-0220,Himachal Pradesh,Kangra,K>NGRA,24,KANGRA,KANGRA,EXACT_NORMALIZED_CANDIDATE_REQUIRES_REVIEW,PENDING,False,NaN
8,APY-GEO-0270,Jharkhand,Saraikela Kharsawan,SARAIKELA-KHARSAWAN,369,SARAIKELA KHARSAWAN,SARAIKELA KHARSAWAN,EXACT_NORMALIZED_CANDIDATE_REQUIRES_REVIEW,PENDING,False,NaN
9,APY-GEO-0320,Madhya Pradesh,Agar-Malwa,AGAR MALWA,735,AGAR MALWA,AGAR MALWA,EXACT_NORMALIZED_CANDIDATE_REQUIRES_REVIEW,PENDING,False,NaN


In [5]:
# Load the geographic reference and coordinate crosswalk.
geo_ref = pd.read_csv(
    project_root / "metadata" / "apy_district_geographic_reference.csv"
)

geo_crosswalk = pd.read_csv(
    project_root / "metadata" / "apy_district_coordinate_crosswalk.csv"
)

# 1. Validate uniqueness of geography IDs.
print("=" * 75)
print("1. GEOGRAPHY ID UNIQUENESS")
print("=" * 75)

print("Reference rows:", len(geo_ref))
print("Reference unique IDs:", geo_ref["apy_geography_id"].nunique())
print("Reference duplicate IDs:", geo_ref["apy_geography_id"].duplicated().sum())

print("\nCrosswalk rows:", len(geo_crosswalk))
print("Crosswalk unique IDs:", geo_crosswalk["apy_geography_id"].nunique())
print("Crosswalk duplicate IDs:", geo_crosswalk["apy_geography_id"].duplicated().sum())

# 2. Check weather geography IDs against the crosswalk.
weather_ids = set(weather_df["apy_geography_id"].dropna().unique())
crosswalk_ids = set(geo_crosswalk["apy_geography_id"].dropna().unique())

print("\n" + "=" * 75)
print("2. WEATHER ID COVERAGE")
print("=" * 75)

print("Unique weather IDs:", len(weather_ids))
print("Weather IDs found in crosswalk:", len(weather_ids & crosswalk_ids))
print("Weather IDs missing from crosswalk:", len(weather_ids - crosswalk_ids))
print("Crosswalk IDs not used by weather:", len(crosswalk_ids - weather_ids))

if weather_ids - crosswalk_ids:
    print("Examples of missing IDs:", sorted(weather_ids - crosswalk_ids)[:10])

# 3. Compare the reference and crosswalk geography labels.
print("\n" + "=" * 75)
print("3. REFERENCE–CROSSWALK CONSISTENCY")
print("=" * 75)

comparison = geo_ref[
    ["apy_geography_id", "State", "District"]
].merge(
    geo_crosswalk[
        ["apy_geography_id", "State", "District"]
    ],
    on="apy_geography_id",
    how="outer",
    suffixes=("_ref", "_crosswalk"),
    indicator=True,
    validate="one_to_one"
)

state_mismatch = (
    comparison["State_ref"].fillna("").str.strip().str.casefold()
    != comparison["State_crosswalk"].fillna("").str.strip().str.casefold()
)

district_mismatch = (
    comparison["District_ref"].fillna("").str.strip().str.casefold()
    != comparison["District_crosswalk"].fillna("").str.strip().str.casefold()
)

print("Rows not present in both files:", (comparison["_merge"] != "both").sum())
print("State label mismatches:", state_mismatch.sum())
print("District label mismatches:", district_mismatch.sum())

if state_mismatch.any() or district_mismatch.any():
    display(
        comparison.loc[
            state_mismatch | district_mismatch,
            [
                "apy_geography_id",
                "State_ref", "State_crosswalk",
                "District_ref", "District_crosswalk"
            ]
        ].head(15)
    )

# 4. Summarize coordinate confidence and synthetic placeholders.
print("\n" + "=" * 75)
print("4. GEOGRAPHIC QUALITY SUMMARY")
print("=" * 75)

print("\nGeography confidence:")
print(geo_crosswalk["geography_confidence"].value_counts(dropna=False).to_string())

print("\nCoordinate source:")
print(geo_crosswalk["coordinate_source"].value_counts(dropna=False).to_string())

print("\nSynthetic coordinate flag:")
print(geo_crosswalk["is_synthetic"].value_counts(dropna=False).to_string())

print("\nMissing coordinates:")
print(
    geo_crosswalk[["latitude", "longitude"]]
    .isna()
    .sum()
    .to_string()
)

# 5. Check whether the APY observations' state-district pairs match the crosswalk.
print("\n" + "=" * 75)
print("5. APY GEOGRAPHY COVERAGE")
print("=" * 75)

apy_locations = apy_df[["State", "District"]].drop_duplicates()

crosswalk_locations = geo_crosswalk[
    ["State", "District", "apy_geography_id"]
].drop_duplicates()

location_check = apy_locations.merge(
    crosswalk_locations,
    on=["State", "District"],
    how="left",
    indicator=True,
    validate="many_to_one"
)

print("Unique APY state-district pairs:", len(apy_locations))
print("Pairs matched to a geography ID:", (location_check["_merge"] == "both").sum())
print("Pairs without a geography ID:", (location_check["_merge"] == "left_only").sum())

if (location_check["_merge"] == "left_only").any():
    print("\nUnmatched APY locations (first 20):")
    display(
        location_check.loc[
            location_check["_merge"] == "left_only",
            ["State", "District"]
        ].head(20)
    )

1. GEOGRAPHY ID UNIQUENESS
Reference rows: 751
Reference unique IDs: 751
Reference duplicate IDs: 0

Crosswalk rows: 751
Crosswalk unique IDs: 751
Crosswalk duplicate IDs: 0

2. WEATHER ID COVERAGE
Unique weather IDs: 640
Weather IDs found in crosswalk: 640
Weather IDs missing from crosswalk: 0
Crosswalk IDs not used by weather: 111

3. REFERENCE–CROSSWALK CONSISTENCY
Rows not present in both files: 0
State label mismatches: 0
District label mismatches: 0

4. GEOGRAPHIC QUALITY SUMMARY

Geography confidence:
geography_confidence
PROVISIONAL_BOUNDARY_MATCH    640
LOW_SYNTHETIC_PLACEHOLDER     108
LOW_APPROXIMATE_LOCATION        3

Coordinate source:
coordinate_source
BOUNDARY_REPRESENTATIVE_POINT          640
SYNTHETIC_STATE_ANCHOR_OFFSET          108
APPROXIMATE_REPRESENTATIVE_LOCATION      3

Synthetic coordinate flag:
is_synthetic
False    640
True     111

Missing coordinates:
latitude     0
longitude    0

5. APY GEOGRAPHY COVERAGE
Unique APY state-district pairs: 751
Pairs matched

In [6]:
# Extract the starting and ending calendar years from APY agricultural years.
apy_year_table = pd.DataFrame({
    "apy_year": sorted(apy_df["Year"].dropna().astype(str).unique())
})

year_parts = apy_year_table["apy_year"].str.extract(
    r"^(?P<start_year>\d{4})-(?P<end_suffix>\d{2})$"
)

if year_parts.isna().any().any():
    raise ValueError(
        "Unexpected APY Year format found. Inspect the Year values before proceeding."
    )

apy_year_table["start_year"] = year_parts["start_year"].astype(int)
apy_year_table["end_year"] = (
    apy_year_table["start_year"] // 100 * 100
    + year_parts["end_suffix"].astype(int)
)

# Compare APY agricultural years with the weather calendar-year coverage.
weather_years = set(weather_df["calendar_year"].unique())

apy_year_table["start_year_available"] = (
    apy_year_table["start_year"].isin(weather_years)
)

apy_year_table["end_year_available"] = (
    apy_year_table["end_year"].isin(weather_years)
)

print("Agricultural-year alignment assessment")
print("=" * 75)
display(apy_year_table)

print("\nCoverage summary")
print("-" * 75)
print(
    "APY years whose starting calendar year exists in weather data:",
    int(apy_year_table["start_year_available"].sum()),
    "of",
    len(apy_year_table)
)
print(
    "APY years whose ending calendar year exists in weather data:",
    int(apy_year_table["end_year_available"].sum()),
    "of",
    len(apy_year_table)
)

print("\nImportant: calendar-year availability does not prove seasonal alignment.")

Agricultural-year alignment assessment


,apy_year,start_year,end_year,start_year_available,end_year_available
0,2013-14,2013,2014,True,True
1,2014-15,2014,2015,True,True
2,2015-16,2015,2016,True,True
3,2016-17,2016,2017,True,True
4,2017-18,2017,2018,True,True
5,2018-19,2018,2019,True,True
6,2019-20,2019,2020,True,True
7,2020-21,2020,2021,True,True
8,2021-22,2021,2022,True,True
9,2022-23,2022,2023,True,True



Coverage summary
---------------------------------------------------------------------------
APY years whose starting calendar year exists in weather data: 12 of 12
APY years whose ending calendar year exists in weather data: 11 of 12

Important: calendar-year availability does not prove seasonal alignment.


In [7]:
# Assess the proposed starting-calendar-year alignment.
# This is diagnostic only; no dataset is merged or modified.

weather_years = set(weather_df["calendar_year"].unique())

apy_year_table["start_year_weather_available"] = (
    apy_year_table["start_year"].isin(weather_years)
)

apy_year_table["end_year_weather_available"] = (
    apy_year_table["end_year"].isin(weather_years)
)

apy_year_table["temporal_alignment_note"] = apy_year_table.apply(
    lambda row: (
        "Starting-year weather available; full-season alignment unverified"
        if row["start_year_weather_available"]
        else "Starting-year weather unavailable"
    ),
    axis=1
)

display(apy_year_table)

print("\nSummary")
print("APY agricultural years:", len(apy_year_table))
print(
    "Starting-year matches:",
    int(apy_year_table["start_year_weather_available"].sum())
)
print(
    "Ending-year matches:",
    int(apy_year_table["end_year_weather_available"].sum())
)
print(
    "Agricultural years without ending-year weather:",
    apy_year_table.loc[
        ~apy_year_table["end_year_weather_available"], "apy_year"
    ].tolist()
)

print(
    "\nDecision pending: verify the source's agricultural-year definition "
    "before selecting the production integration convention."
)

,apy_year,start_year,end_year,start_year_available,end_year_available,start_year_weather_available,end_year_weather_available,temporal_alignment_note
0,2013-14,2013,2014,True,True,True,True,Starting-year weather available; full-season a...
1,2014-15,2014,2015,True,True,True,True,Starting-year weather available; full-season a...
2,2015-16,2015,2016,True,True,True,True,Starting-year weather available; full-season a...
3,2016-17,2016,2017,True,True,True,True,Starting-year weather available; full-season a...
4,2017-18,2017,2018,True,True,True,True,Starting-year weather available; full-season a...
5,2018-19,2018,2019,True,True,True,True,Starting-year weather available; full-season a...
6,2019-20,2019,2020,True,True,True,True,Starting-year weather available; full-season a...
7,2020-21,2020,2021,True,True,True,True,Starting-year weather available; full-season a...
8,2021-22,2021,2022,True,True,True,True,Starting-year weather available; full-season a...
9,2022-23,2022,2023,True,True,True,True,Starting-year weather available; full-season a...



Summary
APY agricultural years: 12
Starting-year matches: 12
Ending-year matches: 11
Agricultural years without ending-year weather: ['2024-25']

Decision pending: verify the source's agricultural-year definition before selecting the production integration convention.


In [8]:
from pathlib import Path

# Search project documentation for the APY agricultural-year definition.
search_terms = [
    "agricultural year",
    "crop year",
    "2013-14",
    "area production yield",
    "APY"
]

documentation_dirs = [
    project_root / "metadata",
    project_root / "docs",
    project_root / "notebooks" / "reports",
    project_root / "data"
]

text_extensions = {".md", ".txt", ".csv", ".json", ".yml", ".yaml"}

print("Searching project documentation for APY year definitions...\n")

for directory in documentation_dirs:
    if not directory.exists():
        continue

    for path in directory.rglob("*"):
        if (
            not path.is_file()
            or path.suffix.lower() not in text_extensions
            or ".ipynb_checkpoints" in path.parts
            or ".git" in path.parts
        ):
            continue

        try:
            content = path.read_text(encoding="utf-8", errors="ignore")
        except OSError:
            continue

        lines = content.splitlines()
        matches = [
            (i + 1, line.strip())
            for i, line in enumerate(lines)
            if any(term.casefold() in line.casefold() for term in search_terms)
        ]

        if matches:
            print("=" * 75)
            print(path.relative_to(project_root))
            for line_number, line in matches[:8]:
                print(f"  Line {line_number}: {line[:250]}")

Searching project documentation for APY year definitions...

metadata\apy_district_coordinate_crosswalk.csv
  Line 1: apy_geography_id,State,District,latitude,longitude,coordinate_source,is_synthetic,geography_confidence,matched_boundary_name,state_lgd,district_lgd,match_notes
  Line 2: APY-GEO-0001,Andaman And Nicobar Islands,Nicobars,6.998715229240884,93.82364983727028,BOUNDARY_REPRESENTATIVE_POINT,False,PROVISIONAL_BOUNDARY_MATCH,NICOBARS,35.0,638.0,Representative point from boundary polygon. Historical district equivalence not y
  Line 3: APY-GEO-0002,Andaman And Nicobar Islands,North And Middle Andaman,13.22835274841933,92.9528772630076,BOUNDARY_REPRESENTATIVE_POINT,False,PROVISIONAL_BOUNDARY_MATCH,NORTH AND MIDDLE ANDAMAN,35.0,639.0,Representative point from boundary polygon. Histor
  Line 4: APY-GEO-0003,Andaman And Nicobar Islands,South Andamans,10.706411725565111,92.48908164320524,BOUNDARY_REPRESENTATIVE_POINT,False,PROVISIONAL_BOUNDARY_MATCH,SOUTH ANDAMANS,35.0,640.0,Represen

In [9]:
# Reuse project_root and Path from earlier notebook cells.

for directory in [
    project_root / "data" / "raw",
    project_root / "metadata",
    project_root / "docs"
]:
    if not directory.exists():
        continue

    print(f"\n{directory.relative_to(project_root)}")

    for path in sorted(directory.iterdir()):
        if path.is_file():
            print(f"  {path.name}")


data\raw
  DES-District-Data-For-2013-14-to-2015-16.csv
  DES-District-Data-For-2016-17-to-2018-19.csv
  DES-District-Data-For-2019-20-to-2021-22.csv
  DES-District-Data-For-2022-23-to-2024-25.csv

metadata
  apy_district_coordinate_crosswalk.csv
  apy_district_geographic_reference.csv
  data_requirements.csv
  district_boundary_provenance.json
  mandi_price_acquisition_audit.json
  mandi_price_source_registry.json
  source_registry.csv
  weather_acquisition_manifest.csv
  weather_source_registry.csv


In [10]:
raw_dir = project_root / "data" / "raw"

for path in sorted(raw_dir.glob("DES-District-Data-For-*.csv")):
    print(f"\n{'=' * 80}\n{path.name}")

    source_df = pd.read_csv(path, nrows=3)
    print("Columns:")
    print(source_df.columns.tolist())
    print("\nSample rows:")
    display(source_df.head(3))


DES-District-Data-For-2013-14-to-2015-16.csv
Columns:
['State', 'District', 'Crop', 'Season', 'Area-2013-14', 'Area-2014-15', 'Area-2015-16', 'Production-2013-14', 'Production-2014-15', 'Production-2015-16', 'Yield-2013-14', 'Yield-2014-15', 'Yield-2015-16']

Sample rows:


,State,District,Crop,Season,Area-2013-14,Area-2014-15,Area-2015-16,Production-2013-14,Production-2014-15,Production-2015-16,Yield-2013-14,Yield-2014-15,Yield-2015-16
0,Andaman And Nicobar Islands,Nicobars,Rice,Kharif,0.0,0.0,0.0,0.0,0.0,0.0,2377.0,2348.0,2240.0
1,Andaman And Nicobar Islands,Nicobars,Rice,Total,0.0,0.0,0.0,0.0,0.0,0.0,2377.0,2348.0,2240.0
2,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,0.0,0.0,0.0,0.0,0.0,0.0,526.0,9167.0,1000.0



DES-District-Data-For-2016-17-to-2018-19.csv
Columns:
['State', 'District', 'Crop', 'Season', 'Area-2016-17', 'Area-2017-18', 'Area-2018-19', 'Production-2016-17', 'Production-2017-18', 'Production-2018-19', 'Yield-2016-17', 'Yield-2017-18', 'Yield-2018-19']

Sample rows:


,State,District,Crop,Season,Area-2016-17,Area-2017-18,Area-2018-19,Production-2016-17,Production-2017-18,Production-2018-19,Yield-2016-17,Yield-2017-18,Yield-2018-19
0,Andaman And Nicobar Islands,Nicobars,Rice,Kharif,0.0,0.0,0.0,0.0,0.0,0.0,1430.0,530.0,1466.0
1,Andaman And Nicobar Islands,Nicobars,Rice,Total,0.0,0.0,0.0,0.0,0.0,0.0,1430.0,530.0,1466.0
2,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,0.0,0.0,0.0,0.0,0.0,0.0,1430.0,530.0,1466.0



DES-District-Data-For-2019-20-to-2021-22.csv
Columns:
['State', 'District', 'Crop', 'Season', 'Area-2019-20', 'Area-2020-21', 'Area-2021-22', 'Production-2019-20', 'Production-2020-21', 'Production-2021-22', 'Yield-2019-20', 'Yield-2020-21', 'Yield-2021-22']

Sample rows:


,State,District,Crop,Season,Area-2019-20,Area-2020-21,Area-2021-22,Production-2019-20,Production-2020-21,Production-2021-22,Yield-2019-20,Yield-2020-21,Yield-2021-22
0,Andaman And Nicobar Islands,Nicobars,Rice,Kharif,0.0,NaN,NaN,0.0,NaN,NaN,1387.0,NaN,NaN
1,Andaman And Nicobar Islands,Nicobars,Rice,Total,0.0,NaN,NaN,0.0,NaN,NaN,1387.0,NaN,NaN
2,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,1133.0



DES-District-Data-For-2022-23-to-2024-25.csv
Columns:
['State', 'District', 'Crop', 'Season', 'Area-2022-23', 'Area-2023-24', 'Area-2024-25', 'Production-2022-23', 'Production-2023-24', 'Production-2024-25', 'Yield-2022-23', 'Yield-2023-24', 'Yield-2024-25']

Sample rows:


,State,District,Crop,Season,Area-2022-23,Area-2023-24,Area-2024-25,Production-2022-23,Production-2023-24,Production-2024-25,Yield-2022-23,Yield-2023-24,Yield-2024-25
0,Andaman And Nicobar Islands,Nicobars,Rice,Kharif,0.0,0.0,0.0,0.0,0.0,0.0,1260.0,2266.0,735.0
1,Andaman And Nicobar Islands,Nicobars,Rice,Total,0.0,0.0,0.0,0.0,0.0,0.0,1260.0,2266.0,735.0
2,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,0.0,0.0,0.0,0.0,0.0,0.0,1260.0,2266.0,735.0


In [11]:
# Reuse the existing project_root and pd variables.

for filename in [
    "source_registry.csv",
    "data_requirements.csv",
    "weather_source_registry.csv",
]:
    path = project_root / "metadata" / filename

    if path.exists():
        print(f"\n{'=' * 70}\n{filename}")
        metadata_df = pd.read_csv(path)
        display(metadata_df)


source_registry.csv


,source_id,dataset_name,organization,official_url,category,access_method,years_available,geographic_coverage,schema_verified,download_verified,coverage_verified,license_verified,integration_verified,status,limitations
0,SRC001,"District-level Area, Production and Yield",Department of Agriculture and Farmers Welfare,https://www.upag.gov.in/primary-estimate-report,Crop statistics,Portal/API; confirm requirements,NaN,NaN,False,False,False,False,False,DISCOVERED,Actual export and data coverage not yet audited
1,SRC002,Current Daily Mandi Prices,Directorate of Marketing and Inspection,https://www.data.gov.in/catalog/current-daily-...,Market prices,Government portal/API/download,NaN,NaN,False,False,False,False,False,DISCOVERED,Historical depth and missingness not yet audited
2,SRC003,IMD Daily Gridded Rainfall,India Meteorological Department,https://www.imdpune.gov.in/cmpg/Griddata/Rainf...,Weather,Official data portal; verify download conditions,NaN,NaN,False,False,False,False,False,DISCOVERED,"File format, access conditions and processing ..."



data_requirements.csv


,category,required_information,intended_use,priority
0,Crop statistics,"Crop area, production, yield",Historical analysis and yield prediction,P0
1,Weather,"Rainfall, temperature, weather extremes",Weather features and agricultural risk analysis,P0
2,Geography,"State and district identifiers, boundaries",Geographic standardization and mapping,P0
3,Market prices,"Commodity, market, date, prices",Price trends and market analysis,P1
4,Soil and soil moisture,Soil properties and moisture observations,Crop suitability and water-stress analysis,P1
5,Satellite observations,Vegetation and crop-condition indicators,Agricultural monitoring and model features,P1
6,Cultivation economics,Cultivation costs and related estimates,Profitability comparisons,P2
7,Supply and trade,"Procurement, stocks, imports and exports",Supply scenarios and market intelligence,P2



weather_source_registry.csv


,source_id,dataset_name,publisher,source_url,candidate_format,candidate_resolution,candidate_unit,candidate_period,coverage_verified,download_verified,schema_verified,license_and_terms_verified,status
0,SRC004,IMD Daily Gridded Rainfall 0.25 Degree,India Meteorological Department,https://www.imdpune.gov.in/cmpg/Griddata/Rainf...,NetCDF,0.25 x 0.25 degree,mm,2013-2024,False,False,False,False,DISCOVERED_NOT_DOWNLOADED


In [12]:
display(apy_year_table)

,apy_year,start_year,end_year,start_year_available,end_year_available,start_year_weather_available,end_year_weather_available,temporal_alignment_note
0,2013-14,2013,2014,True,True,True,True,Starting-year weather available; full-season a...
1,2014-15,2014,2015,True,True,True,True,Starting-year weather available; full-season a...
2,2015-16,2015,2016,True,True,True,True,Starting-year weather available; full-season a...
3,2016-17,2016,2017,True,True,True,True,Starting-year weather available; full-season a...
4,2017-18,2017,2018,True,True,True,True,Starting-year weather available; full-season a...
5,2018-19,2018,2019,True,True,True,True,Starting-year weather available; full-season a...
6,2019-20,2019,2020,True,True,True,True,Starting-year weather available; full-season a...
7,2020-21,2020,2021,True,True,True,True,Starting-year weather available; full-season a...
8,2021-22,2021,2022,True,True,True,True,Starting-year weather available; full-season a...
9,2022-23,2022,2023,True,True,True,True,Starting-year weather available; full-season a...


In [13]:
print("Columns:", apy_year_table.columns.tolist())
print("\nFirst row as a dictionary:")
print(apy_year_table.iloc[0].to_dict())

print("\nData types:")
print(apy_year_table.dtypes)

Columns: ['apy_year', 'start_year', 'end_year', 'start_year_available', 'end_year_available', 'start_year_weather_available', 'end_year_weather_available', 'temporal_alignment_note']

First row as a dictionary:
{'apy_year': '2013-14', 'start_year': 2013, 'end_year': 2014, 'start_year_available': True, 'end_year_available': True, 'start_year_weather_available': True, 'end_year_weather_available': True, 'temporal_alignment_note': 'Starting-year weather available; full-season alignment unverified'}

Data types:
apy_year                        object
start_year                       int64
end_year                         int64
start_year_available              bool
end_year_available                bool
start_year_weather_available      bool
end_year_weather_available        bool
temporal_alignment_note         object
dtype: object


In [14]:
# Reuse project_root and Path from earlier cells.

for path in sorted((project_root / "data" / "raw").glob("DES-District-Data-For-*.csv")):
    print(f"\n{path.name}")
    with path.open("r", encoding="utf-8-sig") as file:
        for _ in range(2):
            line = file.readline()
            if not line:
                break
            print(line.rstrip())


DES-District-Data-For-2013-14-to-2015-16.csv
"State","District","Crop","Season","Area-2013-14","Area-2014-15","Area-2015-16","Production-2013-14","Production-2014-15","Production-2015-16","Yield-2013-14","Yield-2014-15","Yield-2015-16"
"Andaman And Nicobar Islands","Nicobars","Rice","Kharif","0.00","0.00","0.00","0.00","0.00","0.00","2377.00","2348.00","2240.00"

DES-District-Data-For-2016-17-to-2018-19.csv
"State","District","Crop","Season","Area-2016-17","Area-2017-18","Area-2018-19","Production-2016-17","Production-2017-18","Production-2018-19","Yield-2016-17","Yield-2017-18","Yield-2018-19"
"Andaman And Nicobar Islands","Nicobars","Rice","Kharif","0.00","0.00","0.00","0.00","0.00","0.00","1430.00","530.00","1466.00"

DES-District-Data-For-2019-20-to-2021-22.csv
"State","District","Crop","Season","Area-2019-20","Area-2020-21","Area-2021-22","Production-2019-20","Production-2020-21","Production-2021-22","Yield-2019-20","Yield-2020-21","Yield-2021-22"
"Andaman And Nicobar Islands","N

In [15]:
# Create an explicit exploratory mapping; do not imply full-season alignment.

weather_proxy_table = apy_year_table[
    ["apy_year", "start_year", "start_year_weather_available"]
].copy()

weather_proxy_table = weather_proxy_table.rename(
    columns={
        "start_year": "weather_proxy_calendar_year",
        "start_year_weather_available": "weather_proxy_available",
    }
)

weather_proxy_table["alignment_method"] = "STARTING_CALENDAR_YEAR_PROXY"
weather_proxy_table["full_season_alignment_verified"] = False

weather_proxy_table["alignment_limitation"] = (
    "Exploratory proxy only; APY reporting period and crop-specific "
    "weather exposure are not verified."
)

display(weather_proxy_table)

,apy_year,weather_proxy_calendar_year,weather_proxy_available,alignment_method,full_season_alignment_verified,alignment_limitation
0,2013-14,2013,True,STARTING_CALENDAR_YEAR_PROXY,False,Exploratory proxy only; APY reporting period a...
1,2014-15,2014,True,STARTING_CALENDAR_YEAR_PROXY,False,Exploratory proxy only; APY reporting period a...
2,2015-16,2015,True,STARTING_CALENDAR_YEAR_PROXY,False,Exploratory proxy only; APY reporting period a...
3,2016-17,2016,True,STARTING_CALENDAR_YEAR_PROXY,False,Exploratory proxy only; APY reporting period a...
4,2017-18,2017,True,STARTING_CALENDAR_YEAR_PROXY,False,Exploratory proxy only; APY reporting period a...
5,2018-19,2018,True,STARTING_CALENDAR_YEAR_PROXY,False,Exploratory proxy only; APY reporting period a...
6,2019-20,2019,True,STARTING_CALENDAR_YEAR_PROXY,False,Exploratory proxy only; APY reporting period a...
7,2020-21,2020,True,STARTING_CALENDAR_YEAR_PROXY,False,Exploratory proxy only; APY reporting period a...
8,2021-22,2021,True,STARTING_CALENDAR_YEAR_PROXY,False,Exploratory proxy only; APY reporting period a...
9,2022-23,2022,True,STARTING_CALENDAR_YEAR_PROXY,False,Exploratory proxy only; APY reporting period a...


In [16]:
# Reuse the existing apy_df, weather_df, geo_crosswalk,
# and weather_proxy_table variables.

apy_geo_check = apy_df.merge(
    geo_crosswalk[["State", "District", "apy_geography_id"]],
    on=["State", "District"],
    how="left",
    validate="many_to_one",
)

apy_geo_check["start_year"] = (
    apy_geo_check["Year"].str[:4].astype("int64")
)

weather_proxy_check = weather_df.rename(
    columns={"calendar_year": "weather_proxy_calendar_year"}
)

weather_proxy_check = weather_proxy_check.merge(
    weather_proxy_table[
        ["weather_proxy_calendar_year", "weather_proxy_available"]
    ],
    on="weather_proxy_calendar_year",
    how="inner",
    validate="many_to_one",
)

apy_weather_check = apy_geo_check.merge(
    weather_proxy_check,
    left_on=["apy_geography_id", "start_year"],
    right_on=["apy_geography_id", "weather_proxy_calendar_year"],
    how="left",
    indicator=True,
    validate="many_to_one",
)

print("APY rows:", len(apy_geo_check))
print("APY rows with geography ID:", apy_geo_check["apy_geography_id"].notna().sum())
print("APY rows with matched proxy weather:", (apy_weather_check["_merge"] == "both").sum())
print("APY rows without matched proxy weather:", (apy_weather_check["_merge"] == "left_only").sum())
print(
    "Matched APY row percentage:",
    round((apy_weather_check["_merge"] == "both").mean() * 100, 2),
    "%",
)

APY rows: 376272
APY rows with geography ID: 376272
APY rows with matched proxy weather: 320124
APY rows without matched proxy weather: 56148
Matched APY row percentage: 85.08 %


In [17]:
unmatched = apy_weather_check.loc[
    apy_weather_check["_merge"] == "left_only"
].copy()

print("Unmatched rows by APY year:")
display(
    unmatched.groupby("Year")
    .size()
    .rename("unmatched_rows")
    .to_frame()
)

print("\nUnmatched rows by geography ID availability:")
display(
    unmatched["apy_geography_id"]
    .isna()
    .value_counts()
    .rename(index={True: "Missing geography ID", False: "Has geography ID"})
    .to_frame("rows")
)

print("\nDistinct unmatched districts:")
print(
    unmatched[["State", "District", "apy_geography_id"]]
    .drop_duplicates()
    .shape[0]
)

print("\nSample unmatched district records:")
display(
    unmatched[["State", "District", "Year", "apy_geography_id"]]
    .drop_duplicates()
    .head(20)
)

Unmatched rows by APY year:


,unmatched_rows
Year,
2013-14,4679
2014-15,4679
2015-16,4679
2016-17,4679
2017-18,4679
2018-19,4679
2019-20,4679
2020-21,4679
2021-22,4679



Unmatched rows by geography ID availability:


,rows
apy_geography_id,
Has geography ID,56148



Distinct unmatched districts:
111

Sample unmatched district records:


,State,District,Year,apy_geography_id
66,Andhra Pradesh,Alluri Sitharama Raju,2013-14,APY-GEO-0004
1153,Andhra Pradesh,Y.S.R. Kadapa,2013-14,APY-GEO-0029
1465,Arunachal Pradesh,Leparada,2013-14,APY-GEO-0038
1741,Arunachal Pradesh,Papum Pare,2013-14,APY-GEO-0046
2321,Assam,Darrang,2013-14,APY-GEO-0062
2735,Assam,Kamrup,2013-14,APY-GEO-0072
3113,Assam,Sivasagar,2013-14,APY-GEO-0081
3197,Assam,South Salmara Mancachar,2013-14,APY-GEO-0083
3363,Assam,West Karbi Anglong,2013-14,APY-GEO-0087
4603,Bihar,Purbi Champaran,2013-14,APY-GEO-0114


In [18]:
# Reuse existing variables; no new imports needed.

weather_geo_ids = set(weather_df["apy_geography_id"].unique())

unmatched_districts = (
    unmatched[["State", "District", "apy_geography_id"]]
    .drop_duplicates()
    .copy()
)

unmatched_districts["weather_data_exists"] = (
    unmatched_districts["apy_geography_id"].isin(weather_geo_ids)
)

print("Distinct unmatched districts:", len(unmatched_districts))
print(
    "Unmatched districts with weather data:",
    unmatched_districts["weather_data_exists"].sum(),
)
print(
    "Unmatched districts without weather data:",
    (~unmatched_districts["weather_data_exists"]).sum(),
)

print("\nGeographic confidence of unmatched districts:")
display(
    unmatched_districts.merge(
        geo_crosswalk[
            ["apy_geography_id", "geography_confidence", "coordinate_source"]
        ].drop_duplicates("apy_geography_id"),
        on="apy_geography_id",
        how="left",
        validate="one_to_one",
    )
    .groupby(["geography_confidence", "coordinate_source"], dropna=False)
    .size()
    .rename("districts")
    .to_frame()
)

Distinct unmatched districts: 111
Unmatched districts with weather data: 0
Unmatched districts without weather data: 111

Geographic confidence of unmatched districts:


,,districts
geography_confidence,coordinate_source,
LOW_APPROXIMATE_LOCATION,APPROXIMATE_REPRESENTATIVE_LOCATION,3
LOW_SYNTHETIC_PLACEHOLDER,SYNTHETIC_STATE_ANCHOR_OFFSET,108


In [19]:
# Identify APY districts that have no corresponding weather observations.
# These districts were already confirmed to have valid geography IDs,
# so the missing matches are due to absent weather coverage, not failed joins.

weather_coverage_audit = unmatched_districts.copy()

# Add the geographic-confidence metadata so we can distinguish
# synthetic/approximate locations from higher-confidence locations.
weather_coverage_audit = weather_coverage_audit.merge(
    geo_crosswalk[
        [
            "apy_geography_id",
            "geography_confidence",
            "coordinate_source",
            "is_synthetic",
        ]
    ].drop_duplicates("apy_geography_id"),
    on="apy_geography_id",
    how="left",
    validate="one_to_one",
)

# Summarize the uncovered districts by confidence level and coordinate source.
# This audit helps document why weather features must remain missing
# instead of being fabricated or filled using unreliable coordinates.
weather_coverage_summary = (
    weather_coverage_audit
    .groupby(
        ["geography_confidence", "coordinate_source", "is_synthetic"],
        dropna=False,
    )
    .agg(
        district_count=("apy_geography_id", "nunique"),
        states_affected=("State", "nunique"),
    )
    .reset_index()
)

print("Total APY districts without weather coverage:",
      weather_coverage_audit["apy_geography_id"].nunique())

print("\nWeather coverage gap by geographic confidence:")
display(weather_coverage_summary)

print("\nSample of uncovered districts:")
display(weather_coverage_audit.head(10))

Total APY districts without weather coverage: 111

Weather coverage gap by geographic confidence:


,geography_confidence,coordinate_source,is_synthetic,district_count,states_affected
0,LOW_APPROXIMATE_LOCATION,APPROXIMATE_REPRESENTATIVE_LOCATION,True,3,1
1,LOW_SYNTHETIC_PLACEHOLDER,SYNTHETIC_STATE_ANCHOR_OFFSET,True,108,25



Sample of uncovered districts:


,State,District,apy_geography_id,weather_data_exists,geography_confidence,coordinate_source,is_synthetic
0,Andhra Pradesh,Alluri Sitharama Raju,APY-GEO-0004,False,LOW_SYNTHETIC_PLACEHOLDER,SYNTHETIC_STATE_ANCHOR_OFFSET,True
1,Andhra Pradesh,Y.S.R. Kadapa,APY-GEO-0029,False,LOW_SYNTHETIC_PLACEHOLDER,SYNTHETIC_STATE_ANCHOR_OFFSET,True
2,Arunachal Pradesh,Leparada,APY-GEO-0038,False,LOW_SYNTHETIC_PLACEHOLDER,SYNTHETIC_STATE_ANCHOR_OFFSET,True
3,Arunachal Pradesh,Papum Pare,APY-GEO-0046,False,LOW_SYNTHETIC_PLACEHOLDER,SYNTHETIC_STATE_ANCHOR_OFFSET,True
4,Assam,Darrang,APY-GEO-0062,False,LOW_SYNTHETIC_PLACEHOLDER,SYNTHETIC_STATE_ANCHOR_OFFSET,True
5,Assam,Kamrup,APY-GEO-0072,False,LOW_SYNTHETIC_PLACEHOLDER,SYNTHETIC_STATE_ANCHOR_OFFSET,True
6,Assam,Sivasagar,APY-GEO-0081,False,LOW_SYNTHETIC_PLACEHOLDER,SYNTHETIC_STATE_ANCHOR_OFFSET,True
7,Assam,South Salmara Mancachar,APY-GEO-0083,False,LOW_SYNTHETIC_PLACEHOLDER,SYNTHETIC_STATE_ANCHOR_OFFSET,True
8,Assam,West Karbi Anglong,APY-GEO-0087,False,LOW_SYNTHETIC_PLACEHOLDER,SYNTHETIC_STATE_ANCHOR_OFFSET,True
9,Bihar,Purbi Champaran,APY-GEO-0114,False,LOW_SYNTHETIC_PLACEHOLDER,SYNTHETIC_STATE_ANCHOR_OFFSET,True


In [20]:
# Count unique APY district-year combinations.
# Multiple crops and seasons can create many APY rows for the same district-year,
# so counting raw rows would exaggerate the number of independent observations.
apy_district_years = (
    apy_geo_check[
        ["State", "District", "apy_geography_id", "Year", "start_year"]
    ]
    .drop_duplicates()
    .copy()
)

# Check which APY district-years have a matching starting-calendar-year
# weather record. The temporal mapping is still an exploratory proxy,
# not verified full-season alignment.
weather_district_year_keys = weather_proxy_check[
    ["apy_geography_id", "weather_proxy_calendar_year"]
].drop_duplicates()

apy_district_years = apy_district_years.merge(
    weather_district_year_keys,
    left_on=["apy_geography_id", "start_year"],
    right_on=["apy_geography_id", "weather_proxy_calendar_year"],
    how="left",
    indicator=True,
    validate="one_to_one",
)

# Summarize coverage by agricultural year to identify any year-specific gaps.
# Consistent gaps across years would indicate a geographic coverage limitation.
district_year_coverage = (
    apy_district_years
    .groupby("Year")
    .agg(
        apy_districts=("apy_geography_id", "nunique"),
        matched_districts=("_merge", lambda s: (s == "both").sum()),
    )
    .reset_index()
)

district_year_coverage["unmatched_districts"] = (
    district_year_coverage["apy_districts"]
    - district_year_coverage["matched_districts"]
)

district_year_coverage["coverage_pct"] = (
    district_year_coverage["matched_districts"]
    / district_year_coverage["apy_districts"]
    * 100
).round(2)

print("Overall APY district-year combinations:", len(apy_district_years))
print("\nDistrict-year weather coverage by APY year:")
display(district_year_coverage)

Overall APY district-year combinations: 9012

District-year weather coverage by APY year:


,Year,apy_districts,matched_districts,unmatched_districts,coverage_pct
0,2013-14,751,640,111,85.22
1,2014-15,751,640,111,85.22
2,2015-16,751,640,111,85.22
3,2016-17,751,640,111,85.22
4,2017-18,751,640,111,85.22
5,2018-19,751,640,111,85.22
6,2019-20,751,640,111,85.22
7,2020-21,751,640,111,85.22
8,2021-22,751,640,111,85.22
9,2022-23,751,640,111,85.22


In [21]:
# Save the district-year coverage table so downstream analysis can reuse
# the same verified geographic coverage counts without recomputing the join.
coverage_output = (
    project_root
    / "data"
    / "processed"
    / "weather_regional"
    / "apy_weather_proxy_district_year_coverage.csv"
)

coverage_output.parent.mkdir(parents=True, exist_ok=True)
district_year_coverage.to_csv(coverage_output, index=False)

# Save the list of districts without weather records, including their
# geographic-confidence metadata, so the gap is transparent and auditable.
gap_output = (
    project_root
    / "data"
    / "processed"
    / "weather_regional"
    / "apy_districts_without_weather_coverage.csv"
)

weather_coverage_audit.to_csv(gap_output, index=False)

# Print the saved paths and record counts to verify both outputs.
print(f"Coverage summary saved: {coverage_output}")
print(f"Coverage summary rows: {len(district_year_coverage)}")

print(f"\nDistrict gap audit saved: {gap_output}")
print(f"Uncovered districts recorded: {len(weather_coverage_audit)}")

Coverage summary saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\weather_regional\apy_weather_proxy_district_year_coverage.csv
Coverage summary rows: 12

District gap audit saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\weather_regional\apy_districts_without_weather_coverage.csv
Uncovered districts recorded: 111


In [22]:
# Validate that the weather join preserves the original APY row count.
# A many-to-one join is expected: each APY row should match at most one
# weather record for its geography ID and starting-calendar-year proxy.
assert len(apy_weather_check) == len(apy_df), (
    "Row count changed during the weather join; investigate duplicate keys."
)

# Confirm that every APY row retains its original agricultural-year label.
assert apy_weather_check["Year"].notna().all(), (
    "Some APY rows lost their agricultural-year label."
)

# Check uniqueness of the weather key used in the join.
# Duplicate weather keys could cause unintended row multiplication.
weather_key_duplicates = weather_proxy_check.duplicated(
    subset=["apy_geography_id", "weather_proxy_calendar_year"]
).sum()

print("APY rows preserved:", len(apy_weather_check) == len(apy_df))
print("Weather duplicate join keys:", weather_key_duplicates)
print("Matched APY rows:", (apy_weather_check["_merge"] == "both").sum())
print("Unmatched APY rows:", (apy_weather_check["_merge"] == "left_only").sum())

assert weather_key_duplicates == 0, (
    "Weather join keys are not unique; resolve duplicates before proceeding."
)

APY rows preserved: True
Weather duplicate join keys: 0
Matched APY rows: 320124
Unmatched APY rows: 56148


In [23]:
# Audit missing values separately for APY rows with and without weather matches.
# This helps distinguish missing agricultural measurements from missing weather coverage.

# Select the APY measurement columns relevant to downstream analysis.
apy_measurement_cols = [
    col for col in ["Area", "Production", "Yield"]
    if col in apy_weather_check.columns
]

# Summarize agricultural measurement missingness by weather-join status.
measurement_missingness = (
    apy_weather_check
    .assign(
        weather_match=apy_weather_check["_merge"].map({
            "both": "Matched",
            "left_only": "Unmatched"
        })
    )
    .groupby("weather_match")[apy_measurement_cols]
    .agg(["count", lambda series: series.isna().sum()])
)

# Give the aggregate columns clear names for interpretation.
measurement_missingness.columns = [
    f"{column}_{statistic}"
    for column, statistic in measurement_missingness.columns
]

print("Agricultural measurement missingness by weather-match status:")
display(measurement_missingness)

# Report missingness in weather variables for matched rows only.
# Unmatched rows naturally have missing weather fields because no weather record joined.
weather_cols = [
    col for col in weather_df.columns
    if col not in ["apy_geography_id", "calendar_year"]
    and col in apy_weather_check.columns
]

matched_rows = apy_weather_check.loc[
    apy_weather_check["_merge"] == "both"
]

weather_missingness = matched_rows[weather_cols].isna().sum().sort_values(
    ascending=False
)

print("\nMissing weather values among matched APY rows:")
display(weather_missingness.to_frame(name="missing_count"))

Agricultural measurement missingness by weather-match status:


,Area_count,Area_<lambda_0>,Production_count,Production_<lambda_0>,Yield_count,Yield_<lambda_0>
weather_match,,,,,,
Matched,242765,77359,242154,77970,242151,77973
Unmatched,39324,16824,39088,17060,39088,17060



Missing weather values among matched APY rows:


,missing_count
tmin_mean_c,0
tmin_min_c,0
tmax_mean_c,0
tmax_max_c,0
precipitation_total_mm,0
precipitation_mean_mm_day,0
precipitation_max_mm_day,0
tmin_valid_days,0
tmax_valid_days,0
precipitation_valid_days,0


In [25]:
# Calculate missingness percentages only for weather-match groups that contain rows.
# observed=True excludes unused categorical groups and avoids division by zero.

measurement_cols = [
    col for col in ["Area", "Production", "Yield"]
    if col in apy_weather_check.columns
]

missingness_rows = []

for status, group in apy_weather_check.groupby(
    "_merge",
    dropna=False,
    observed=True
):
    # Defensive check: never calculate percentages for an empty group.
    if group.empty:
        continue

    for col in measurement_cols:
        missing_count = int(group[col].isna().sum())

        missingness_rows.append({
            "weather_match_status": status,
            "measurement": col,
            "total_rows": len(group),
            "missing_rows": missing_count,
            "missing_pct": round(
                100 * missing_count / len(group), 2
            )
        })

measurement_missingness_pct = pd.DataFrame(missingness_rows)

print("APY measurement missingness percentages:")
display(
    measurement_missingness_pct.sort_values(
        ["weather_match_status", "measurement"]
    ).reset_index(drop=True)
)

# Count joint missingness patterns across Area, Production, and Yield.
# True indicates a missing value; each row represents one unique pattern.
apy_missing_patterns = (
    apy_weather_check[measurement_cols]
    .isna()
    .value_counts(dropna=False)
    .rename("row_count")
    .reset_index()
)

print("\nJoint missingness patterns (True means missing):")
display(apy_missing_patterns)

APY measurement missingness percentages:


,weather_match_status,measurement,total_rows,missing_rows,missing_pct
0,both,Area,320124,77359,24.17
1,both,Production,320124,77970,24.36
2,both,Yield,320124,77973,24.36
3,left_only,Area,56148,16824,29.96
4,left_only,Production,56148,17060,30.38
5,left_only,Yield,56148,17060,30.38



Joint missingness patterns (True means missing):


,Area,Production,Yield,row_count
0,False,False,False,281239
1,True,True,True,94180
2,False,True,True,850
3,True,False,True,3


In [26]:
# Count complete agricultural records and records usable for weather-linked analysis.
# Do not impute missing measurements or convert missing values to zero.

measurement_cols = ["Area", "Production", "Yield"]

# A complete APY record has all three agricultural measurements present.
apy_weather_check["apy_measurements_complete"] = (
    apy_weather_check[measurement_cols].notna().all(axis=1)
)

# A weather-linked complete record must also have a matched weather observation.
apy_weather_check["weather_and_apy_complete"] = (
    apy_weather_check["apy_measurements_complete"]
    & apy_weather_check["_merge"].eq("both")
)

# Summarize complete and incomplete records by weather-match status.
complete_record_summary = (
    apy_weather_check
    .groupby("_merge", observed=True)
    .agg(
        total_rows=("Year", "size"),
        complete_apy_rows=("apy_measurements_complete", "sum"),
        complete_weather_linked_rows=("weather_and_apy_complete", "sum")
    )
    .reset_index()
)

complete_record_summary["complete_apy_pct"] = (
    100 * complete_record_summary["complete_apy_rows"]
    / complete_record_summary["total_rows"]
).round(2)

print("Complete APY records by weather-match status:")
display(complete_record_summary)

# Report overall counts for downstream analysis planning.
print("\nOverall record counts:")
print("Total APY rows:", len(apy_weather_check))
print(
    "Rows with all three APY measurements:",
    int(apy_weather_check["apy_measurements_complete"].sum())
)
print(
    "Rows with complete APY measurements and matched weather:",
    int(apy_weather_check["weather_and_apy_complete"].sum())
)

Complete APY records by weather-match status:


,_merge,total_rows,complete_apy_rows,complete_weather_linked_rows,complete_apy_pct
0,left_only,56148,39088,0,69.62
1,both,320124,242151,242151,75.64



Overall record counts:
Total APY rows: 376272
Rows with all three APY measurements: 281239
Rows with complete APY measurements and matched weather: 242151


In [27]:
# Build the integrated APY-weather dataset for exploratory analysis.
# Preserve every APY row, including districts without weather coverage.
# Weather is linked using the starting-calendar-year proxy; full-season
# temporal alignment has not been verified, so the dataset must be labeled.

# Copy the validated join so the integration artifact can be prepared safely.
apy_weather_integrated = apy_weather_check.copy()

# Record the weather-linkage method and its known temporal limitation.
apy_weather_integrated["weather_alignment_method"] = (
    "STARTING_CALENDAR_YEAR_PROXY"
)
apy_weather_integrated["weather_full_season_alignment_verified"] = False

# Make weather coverage explicit for downstream analysis and filtering.
apy_weather_integrated["weather_available"] = (
    apy_weather_integrated["_merge"] == "both"
)

# Preserve a readable match status rather than relying only on the merge indicator.
apy_weather_integrated["weather_match_status"] = (
    apy_weather_integrated["_merge"].map({
        "both": "MATCHED",
        "left_only": "NO_WEATHER_COVERAGE"
    })
)

# Save the integrated exploratory dataset and a compact coverage summary.
integrated_dir = project_root / "data" / "processed" / "weather_regional"
integrated_dir.mkdir(parents=True, exist_ok=True)

integrated_path = integrated_dir / "apy_weather_integrated_exploratory.csv"
apy_weather_integrated.to_csv(integrated_path, index=False)

integration_summary = pd.DataFrame([{
    "apy_rows": len(apy_weather_integrated),
    "matched_rows": int(apy_weather_integrated["weather_available"].sum()),
    "unmatched_rows": int((~apy_weather_integrated["weather_available"]).sum()),
    "complete_apy_rows": int(
        apy_weather_integrated["apy_measurements_complete"].sum()
    ),
    "complete_apy_and_weather_rows": int(
        apy_weather_integrated["weather_and_apy_complete"].sum()
    ),
    "weather_alignment_method": "STARTING_CALENDAR_YEAR_PROXY",
    "full_season_alignment_verified": False
}])

summary_path = integrated_dir / "apy_weather_integration_summary.csv"
integration_summary.to_csv(summary_path, index=False)

print("Integrated dataset saved:", integrated_path)
print("Integration summary saved:", summary_path)
display(integration_summary)

# Confirm that integration did not drop APY records.
assert len(apy_weather_integrated) == len(apy_df), (
    "Integrated dataset does not preserve all original APY rows."
)
assert apy_weather_integrated["weather_available"].sum() == 320124
assert (~apy_weather_integrated["weather_available"]).sum() == 56148

print("Validation passed: all original APY rows are preserved.")

Integrated dataset saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\weather_regional\apy_weather_integrated_exploratory.csv
Integration summary saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\weather_regional\apy_weather_integration_summary.csv


,apy_rows,matched_rows,unmatched_rows,complete_apy_rows,complete_apy_and_weather_rows,weather_alignment_method,full_season_alignment_verified
0,376272,320124,56148,281239,242151,STARTING_CALENDAR_YEAR_PROXY,False


Validation passed: all original APY rows are preserved.


In [28]:
# Inspect the integrated dataset's schema before selecting analysis features.
# This avoids assuming column names or accidentally using metadata as predictors.

print("Integrated dataset shape:", apy_weather_integrated.shape)

print("\nColumn names and data types:")
display(
    pd.DataFrame({
        "column": apy_weather_integrated.columns,
        "dtype": apy_weather_integrated.dtypes.astype(str).values,
        "missing_count": apy_weather_integrated.isna().sum().values,
        "missing_pct": (
            100 * apy_weather_integrated.isna().mean()
        ).round(2).values
    }).sort_values("column").reset_index(drop=True)
)

# Print likely analysis fields so we can identify the exact agricultural dimensions.
candidate_terms = [
    "crop", "season", "year", "state", "district",
    "area", "production", "yield", "rain", "precipitation",
    "temp", "tmin", "tmax", "weather", "geography"
]

candidate_columns = [
    col for col in apy_weather_integrated.columns
    if any(term in col.lower() for term in candidate_terms)
]

print("\nPotential crop, geography, time, and weather fields:")
print(candidate_columns)

# Inspect a few rows to verify how the joined records are represented.
display(apy_weather_integrated[candidate_columns].head(5))

Integrated dataset shape: (376272, 39)

Column names and data types:


,column,dtype,missing_count,missing_pct
0,Area,float64,94183,25.03
1,Crop,object,0,0.00
2,Crop_Category,object,0,0.00
3,District,object,0,0.00
4,Production,float64,95030,25.26
5,QC_AreaPositive-ProductionZero,bool,0,0.00
6,QC_AreaZero-ProductionPositive,bool,0,0.00
7,QC_Extreme_Annual_Max_Temperature,object,56148,14.92
8,QC_Extreme_Annual_Min_Temperature,object,56148,14.92
9,QC_Extreme_Daily_Precipitation,object,56148,14.92



Potential crop, geography, time, and weather fields:
['State', 'District', 'Crop', 'Season', 'Crop_Category', 'Year', 'Area', 'Production', 'Yield', 'QC_AreaZero-ProductionPositive', 'QC_AreaPositive-ProductionZero', 'QC_PositiveYieldWithoutPositiveAreaProduction', 'apy_geography_id', 'start_year', 'weather_proxy_calendar_year', 'tmin_mean_c', 'tmin_min_c', 'tmax_mean_c', 'tmax_max_c', 'precipitation_total_mm', 'precipitation_mean_mm_day', 'precipitation_max_mm_day', 'tmin_valid_days', 'tmax_valid_days', 'precipitation_valid_days', 'QC_Extreme_Annual_Min_Temperature', 'QC_Extreme_Annual_Max_Temperature', 'QC_High_Annual_Precipitation', 'QC_Extreme_Daily_Precipitation', 'QC_Long_District_Grid_Distance', 'weather_proxy_available', 'weather_and_apy_complete', 'weather_alignment_method', 'weather_full_season_alignment_verified', 'weather_available', 'weather_match_status']


,State,District,Crop,Season,Crop_Category,Year,Area,Production,Yield,QC_AreaZero-ProductionPositive,...,QC_Extreme_Annual_Max_Temperature,QC_High_Annual_Precipitation,QC_Extreme_Daily_Precipitation,QC_Long_District_Grid_Distance,weather_proxy_available,weather_and_apy_complete,weather_alignment_method,weather_full_season_alignment_verified,weather_available,weather_match_status
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,Grouping — Verify Source Definition,2013-14,0.0,0.0,2377.0,False,...,False,False,False,False,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,Grouping — Verify Source Definition,2013-14,0.0,0.0,526.0,False,...,False,False,False,False,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,Grouping — Verify Source Definition,2013-14,0.0,0.0,923.0,False,...,False,False,False,False,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED
3,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,Individual Crop Candidate,2013-14,0.0,0.0,526.0,False,...,False,False,False,False,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED
4,Andaman And Nicobar Islands,Nicobars,Maize,Total,Individual Crop Candidate,2013-14,0.0,0.0,526.0,False,...,False,False,False,False,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED


In [29]:
# Audit crop and season dimensions before conducting crop-weather analysis.
# Aggregate groupings may coexist with individual crops, so inspect the
# source-provided categories rather than assuming every Crop is individual.

print("Crop category distribution:")
display(
    apy_weather_integrated["Crop_Category"]
    .value_counts(dropna=False)
    .rename_axis("Crop_Category")
    .reset_index(name="row_count")
)

print("\nMost frequent crop labels:")
display(
    apy_weather_integrated["Crop"]
    .value_counts(dropna=False)
    .head(25)
    .rename_axis("Crop")
    .reset_index(name="row_count")
)

print("\nSeason distribution:")
display(
    apy_weather_integrated["Season"]
    .value_counts(dropna=False)
    .rename_axis("Season")
    .reset_index(name="row_count")
)

# Summarize yield availability and value ranges by source-provided crop category.
# Negative and zero values are retained here for inspection, not silently removed.
yield_audit = (
    apy_weather_integrated
    .groupby("Crop_Category", dropna=False, observed=True)
    .agg(
        total_rows=("Yield", "size"),
        nonmissing_yield_rows=("Yield", "count"),
        zero_yield_rows=("Yield", lambda s: int(s.eq(0).sum())),
        negative_yield_rows=("Yield", lambda s: int(s.lt(0).sum())),
        distinct_crops=("Crop", "nunique")
    )
    .reset_index()
)

yield_audit["yield_available_pct"] = (
    100 * yield_audit["nonmissing_yield_rows"] / yield_audit["total_rows"]
).round(2)

print("\nYield availability by crop category:")
display(yield_audit)

Crop category distribution:


,Crop_Category,row_count
0,Individual Crop Candidate,226560
1,Grouping — Verify Source Definition,107640
2,Aggregate Total — Exclude from Individual-Crop...,42072



Most frequent crop labels:


,Crop,row_count
0,Total Food Grains,33108
1,Cereals,32124
2,Nutri/Coarse Cereals,30360
3,Maize,27804
4,Rice,25656
5,Moong,24888
6,Other Pulses,23520
7,Urad,23136
8,Shree Anna /Nutri Cereals,21636
9,Tur,17448



Season distribution:


,Season,row_count
0,Total,143028
1,Kharif,106692
2,Rabi,88704
3,Summer,37848



Yield availability by crop category:


,Crop_Category,total_rows,nonmissing_yield_rows,zero_yield_rows,negative_yield_rows,distinct_crops,yield_available_pct
0,Aggregate Total — Exclude from Individual-Crop...,42072,37110,14,0,2,88.21
1,Grouping — Verify Source Definition,107640,84528,41,0,4,78.53
2,Individual Crop Candidate,226560,159601,362,0,13,70.45


In [30]:
# Profile individual-crop candidates for exploratory crop-weather analysis.
# Aggregate totals and unverified groupings are excluded from this profile,
# but the integrated source dataset itself remains unchanged.

candidate_mask = apy_weather_integrated["Crop_Category"].eq(
    "Individual Crop Candidate"
)

crop_candidates = apy_weather_integrated.loc[candidate_mask].copy()

# Summarize sample size, geography, time coverage, and measurement availability.
crop_profile = (
    crop_candidates
    .groupby("Crop", dropna=False, observed=True)
    .agg(
        total_rows=("Crop", "size"),
        complete_apy_rows=("apy_measurements_complete", "sum"),
        complete_weather_linked_rows=("weather_and_apy_complete", "sum"),
        districts=("apy_geography_id", "nunique"),
        states=("State", "nunique"),
        apy_years=("Year", "nunique"),
        seasons=("Season", "nunique")
    )
    .reset_index()
)

# Calculate the share of each crop's records suitable for complete APY-weather analysis.
crop_profile["complete_weather_linked_pct"] = (
    100 * crop_profile["complete_weather_linked_rows"]
    / crop_profile["total_rows"]
).round(2)

# Sort by usable sample size to identify practical candidates for further investigation.
crop_profile = crop_profile.sort_values(
    ["complete_weather_linked_rows", "districts"],
    ascending=False
).reset_index(drop=True)

print("Individual-crop candidate profile:")
display(crop_profile)

# Inspect data-quality flags for these candidate records.
# These flags are not automatically grounds for exclusion; they need interpretation.
quality_flag_cols = [
    col for col in crop_candidates.columns
    if col.startswith("QC_")
]

print("\nQuality-flag counts among individual-crop candidates:")
display(
    crop_candidates[quality_flag_cols]
    .fillna(False)
    .astype(bool)
    .sum()
    .sort_values(ascending=False)
    .rename_axis("quality_flag")
    .reset_index(name="flagged_rows")
)

Individual-crop candidate profile:


,Crop,total_rows,complete_apy_rows,complete_weather_linked_rows,districts,states,apy_years,seasons,complete_weather_linked_pct
0,Maize,27804,21067,18166,735,32,12,4,65.34
1,Rice,25656,20679,17082,720,33,12,4,66.58
2,Moong,24888,17423,15029,690,27,12,4,60.39
3,Urad,23136,16791,14608,669,29,12,4,63.14
4,Wheat,15216,12607,11090,628,30,12,4,72.88
5,Tur,17448,12598,11080,665,28,12,3,63.50
6,Gram,15468,11815,10351,631,26,12,3,66.92
7,Jowar,15036,9954,8622,504,23,12,4,57.34
8,Bajra,14376,8822,7767,498,24,12,4,54.03
9,Lentil,11364,8241,7383,468,23,12,4,64.97



Quality-flag counts among individual-crop candidates:


C:\Users\gsr58\AppData\Local\Temp\ipykernel_16404\3214845400.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,quality_flag,flagged_rows
0,QC_Long_District_Grid_Distance,85980
1,QC_PositiveYieldWithoutPositiveAreaProduction,68159
2,QC_Extreme_Annual_Max_Temperature,12871
3,QC_AreaZero-ProductionPositive,5225
4,QC_AreaPositive-ProductionZero,4830
5,QC_Extreme_Daily_Precipitation,197
6,QC_Extreme_Annual_Min_Temperature,166
7,QC_High_Annual_Precipitation,156
8,QC_NegativeMeasurements,0


In [31]:
# Create a Rice-only analysis subset without modifying the integrated dataset.
# Keep only rows with complete APY measurements and matched weather.
# This is exploratory analysis using a starting-calendar-year weather proxy,
# not verified full-season weather or evidence of causation.

rice_analysis = apy_weather_integrated.loc[
    apy_weather_integrated["Crop"].eq("Rice")
    & apy_weather_integrated["Crop_Category"].eq("Individual Crop Candidate")
    & apy_weather_integrated["weather_and_apy_complete"]
].copy()

print("Rice analysis subset shape:", rice_analysis.shape)
print("Distinct districts:", rice_analysis["apy_geography_id"].nunique())
print("Distinct states:", rice_analysis["State"].nunique())
print("Agricultural years:", sorted(rice_analysis["Year"].dropna().unique()))

# Inspect yield distribution before deciding whether any records require exclusion.
print("\nRice yield distribution:")
display(
    rice_analysis["Yield"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .to_frame(name="Yield")
)

# Inspect yield and production consistency flags in the Rice subset.
rice_quality_cols = [
    "QC_NegativeMeasurements",
    "QC_AreaZero-ProductionPositive",
    "QC_AreaPositive-ProductionZero",
    "QC_PositiveYieldWithoutPositiveAreaProduction",
    "QC_Long_District_Grid_Distance"
]

rice_quality_cols = [
    col for col in rice_quality_cols if col in rice_analysis.columns
]

print("\nRice quality-flag counts:")
display(
    rice_analysis[rice_quality_cols]
    .eq(True)
    .sum()
    .rename_axis("quality_flag")
    .reset_index(name="flagged_rows")
)

# Count zero and negative yields explicitly; these can affect summaries and models.
print("\nRice yield edge cases:")
print("Zero yield rows:", int(rice_analysis["Yield"].eq(0).sum()))
print("Negative yield rows:", int(rice_analysis["Yield"].lt(0).sum()))
print("Missing yield rows:", int(rice_analysis["Yield"].isna().sum()))

Rice analysis subset shape: (17082, 39)
Distinct districts: 614
Distinct states: 31
Agricultural years: ['2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']

Rice yield distribution:


,Yield
count,17082.000000
mean,2527.292296
std,1077.333706
min,0.000000
1%,414.620000
5%,1016.000000
25%,1842.000000
50%,2479.000000
75%,3135.000000
95%,4246.900000



Rice quality-flag counts:


,quality_flag,flagged_rows
0,QC_NegativeMeasurements,0
1,QC_AreaZero-ProductionPositive,379
2,QC_AreaPositive-ProductionZero,57
3,QC_PositiveYieldWithoutPositiveAreaProduction,1633
4,QC_Long_District_Grid_Distance,7506



Rice yield edge cases:
Zero yield rows: 13
Negative yield rows: 0
Missing yield rows: 0


In [32]:
# Inspect the highest Rice yields to investigate extreme values.
# Retain the original records and show the geographic and weather QC context.
# The weather values are still based on the unverified starting-year proxy.

rice_inspection_cols = [
    "State",
    "District",
    "Year",
    "Season",
    "Crop",
    "Area",
    "Production",
    "Yield",
    "tmin_mean_c",
    "tmax_mean_c",
    "precipitation_total_mm",
    "QC_AreaZero-ProductionPositive",
    "QC_AreaPositive-ProductionZero",
    "QC_PositiveYieldWithoutPositiveAreaProduction",
    "QC_Long_District_Grid_Distance",
    "weather_alignment_method",
    "weather_full_season_alignment_verified"
]

# Show the 20 highest-yield records without modifying or filtering the source dataset.
rice_high_yield_records = (
    rice_analysis
    .sort_values("Yield", ascending=False)
    [rice_inspection_cols]
    .head(20)
    .reset_index(drop=True)
)

print("Top 20 Rice yields for record-level inspection:")
display(rice_high_yield_records)

# Compare the yield distribution for records with and without the long-distance flag.
# This helps assess whether geographic uncertainty is associated with different yields.
rice_geo_flag_summary = (
    rice_analysis
    .groupby("QC_Long_District_Grid_Distance", observed=True)
    .agg(
        rows=("Yield", "size"),
        median_yield=("Yield", "median"),
        mean_yield=("Yield", "mean"),
        p95_yield=("Yield", lambda s: s.quantile(0.95)),
        max_yield=("Yield", "max")
    )
    .reset_index()
)

print("\nYield distribution by district-grid-distance flag:")
display(rice_geo_flag_summary)

Top 20 Rice yields for record-level inspection:


,State,District,Year,Season,Crop,Area,Production,Yield,tmin_mean_c,tmax_mean_c,precipitation_total_mm,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_PositiveYieldWithoutPositiveAreaProduction,QC_Long_District_Grid_Distance,weather_alignment_method,weather_full_season_alignment_verified
0,Goa,North Goa,2022-23,Kharif,Rice,0.01,0.47,44815.0,25.184575,29.703178,2369.19,False,False,False,True,STARTING_CALENDAR_YEAR_PROXY,False
1,Goa,South Goa,2022-23,Kharif,Rice,0.01,0.47,39221.0,25.184575,29.703178,2369.19,False,False,False,True,STARTING_CALENDAR_YEAR_PROXY,False
2,Goa,North Goa,2022-23,Total,Rice,0.03,0.54,16150.0,25.184575,29.703178,2369.19,False,False,False,True,STARTING_CALENDAR_YEAR_PROXY,False
3,Himachal Pradesh,Bilaspur,2020-21,Total,Rice,0.01,0.12,12880.0,13.030519,23.376366,1243.82,False,False,False,False,STARTING_CALENDAR_YEAR_PROXY,False
4,Himachal Pradesh,Bilaspur,2020-21,Kharif,Rice,0.01,0.12,12880.0,13.030519,23.376366,1243.82,False,False,False,False,STARTING_CALENDAR_YEAR_PROXY,False
5,Goa,South Goa,2022-23,Total,Rice,0.08,0.66,8362.0,25.184575,29.703178,2369.19,False,False,False,True,STARTING_CALENDAR_YEAR_PROXY,False
6,Arunachal Pradesh,Kra Daadi,2015-16,Total,Rice,0.00,0.01,7177.0,11.370274,19.195507,2340.50,True,False,True,False,STARTING_CALENDAR_YEAR_PROXY,False
7,Arunachal Pradesh,Kra Daadi,2015-16,Kharif,Rice,0.00,0.01,7177.0,11.370274,19.195507,2340.50,True,False,True,False,STARTING_CALENDAR_YEAR_PROXY,False
8,Arunachal Pradesh,Kurung Kumey,2015-16,Total,Rice,0.00,0.01,7000.0,5.874904,13.513890,1357.39,True,False,True,False,STARTING_CALENDAR_YEAR_PROXY,False
9,Arunachal Pradesh,Kurung Kumey,2015-16,Kharif,Rice,0.00,0.01,7000.0,5.874904,13.513890,1357.39,True,False,True,False,STARTING_CALENDAR_YEAR_PROXY,False



Yield distribution by district-grid-distance flag:


,QC_Long_District_Grid_Distance,rows,median_yield,mean_yield,p95_yield,max_yield
0,False,9576,2526.5,2563.518066,4296.0,12880.0
1,True,7506,2421.0,2481.076206,4155.5,44815.0


In [33]:
# Investigate extreme Rice yields and measurement consistency without deleting records.
# Yield values may be derived from area and production, but the source units and
# formula must be confirmed before treating that relationship as definitive.

# Compare the highest-yield records with the main distribution.
yield_thresholds = [0, 1000, 3000, 5000, 10000]

yield_threshold_summary = pd.DataFrame([
    {
        "threshold_exclusive": threshold,
        "rows_above_threshold": int(
            rice_analysis["Yield"].gt(threshold).sum()
        ),
        "pct_of_rice_subset": round(
            100 * rice_analysis["Yield"].gt(threshold).mean(), 3
        )
    }
    for threshold in yield_thresholds
])

print("Rice yield threshold summary:")
display(yield_threshold_summary)

# Inspect records with very high yields or known area-production consistency flags.
# Keep all flagged records visible so their context is not lost.
rice_suspect_records = rice_analysis.loc[
    rice_analysis["Yield"].gt(10000)
    | rice_analysis["QC_AreaZero-ProductionPositive"].eq(True)
    | rice_analysis["QC_AreaPositive-ProductionZero"].eq(True)
    | rice_analysis["QC_PositiveYieldWithoutPositiveAreaProduction"].eq(True)
].copy()

rice_suspect_cols = [
    "State", "District", "Year", "Season",
    "Area", "Production", "Yield",
    "QC_AreaZero-ProductionPositive",
    "QC_AreaPositive-ProductionZero",
    "QC_PositiveYieldWithoutPositiveAreaProduction",
    "QC_Long_District_Grid_Distance"
]

print("Suspect Rice records:", len(rice_suspect_records))
display(
    rice_suspect_records[rice_suspect_cols]
    .sort_values("Yield", ascending=False)
    .head(30)
)

# Summarize measurement consistency flags without assuming they invalidate yield.
print("\nConsistency-flag overlap:")
display(
    rice_suspect_records[
        [
            "QC_AreaZero-ProductionPositive",
            "QC_AreaPositive-ProductionZero",
            "QC_PositiveYieldWithoutPositiveAreaProduction"
        ]
    ]
    .value_counts(dropna=False)
    .rename("row_count")
    .reset_index()
)

Rice yield threshold summary:


,threshold_exclusive,rows_above_threshold,pct_of_rice_subset
0,0,17069,99.924
1,1000,16265,95.217
2,3000,4919,28.796
3,5000,165,0.966
4,10000,5,0.029


Suspect Rice records: 1638


,State,District,Year,Season,Area,Production,Yield,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_PositiveYieldWithoutPositiveAreaProduction,QC_Long_District_Grid_Distance
288727,Goa,North Goa,2022-23,Kharif,0.01,0.47,44815.0,False,False,False,True
288745,Goa,South Goa,2022-23,Kharif,0.01,0.47,39221.0,False,False,False,True
288729,Goa,North Goa,2022-23,Total,0.03,0.54,16150.0,False,False,False,True
228314,Himachal Pradesh,Bilaspur,2020-21,Total,0.01,0.12,12880.0,False,False,False,False
228313,Himachal Pradesh,Bilaspur,2020-21,Kharif,0.01,0.12,12880.0,False,False,False,False
64137,Arunachal Pradesh,Kra Daadi,2015-16,Total,0.00,0.01,7177.0,True,False,True,False
64135,Arunachal Pradesh,Kra Daadi,2015-16,Kharif,0.00,0.01,7177.0,True,False,True,False
64165,Arunachal Pradesh,Kurung Kumey,2015-16,Total,0.00,0.01,7000.0,True,False,True,False
64163,Arunachal Pradesh,Kurung Kumey,2015-16,Kharif,0.00,0.01,7000.0,True,False,True,False
130573,Chandigarh,Chandigarh,2017-18,Kharif,0.00,0.00,5500.0,False,False,True,True



Consistency-flag overlap:


,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_PositiveYieldWithoutPositiveAreaProduction,row_count
0,False,False,True,1197
1,True,False,True,379
2,False,True,True,57
3,False,False,False,5


In [34]:
# Quantify Rice data-quality issues before deciding which records can be
# included in exploratory yield-weather analysis.
# These are provisional screening rules, not claims that records are invalid.

rice_quality_summary = pd.DataFrame([
    {
        "check": "All selected Rice records",
        "rows": len(rice_analysis)
    },
    {
        "check": "Yield above 5000",
        "rows": int(rice_analysis["Yield"].gt(5000).sum())
    },
    {
        "check": "Yield above 10000",
        "rows": int(rice_analysis["Yield"].gt(10000).sum())
    },
    {
        "check": "Zero yield",
        "rows": int(rice_analysis["Yield"].eq(0).sum())
    },
    {
        "check": "Zero area and positive production",
        "rows": int(
            rice_analysis["QC_AreaZero-ProductionPositive"].eq(True).sum()
        )
    },
    {
        "check": "Positive area and zero production",
        "rows": int(
            rice_analysis["QC_AreaPositive-ProductionZero"].eq(True).sum()
        )
    },
    {
        "check": "Positive yield without positive area and production",
        "rows": int(
            rice_analysis[
                "QC_PositiveYieldWithoutPositiveAreaProduction"
            ].eq(True).sum()
        )
    },
    {
        "check": "Long district-to-weather-grid distance",
        "rows": int(
            rice_analysis["QC_Long_District_Grid_Distance"].eq(True).sum()
        )
    }
])

rice_quality_summary["pct_of_rice_subset"] = (
    100 * rice_quality_summary["rows"] / len(rice_analysis)
).round(2)

display(rice_quality_summary)

# Create separate flags for measurement consistency and extreme yield.
# Do not use weather-location confidence as a yield-validity rule.
rice_analysis["rice_measurement_flag"] = (
    rice_analysis["QC_AreaZero-ProductionPositive"].eq(True)
    | rice_analysis["QC_AreaPositive-ProductionZero"].eq(True)
    | rice_analysis[
        "QC_PositiveYieldWithoutPositiveAreaProduction"
    ].eq(True)
)

rice_analysis["rice_extreme_yield_flag"] = (
    rice_analysis["Yield"].gt(10000)
)

# Provisional screen only: exclude measurement-flagged rows and yields
# above 10000 from the strict exploratory subset. Retain the original data.
rice_analysis["rice_strict_screen_pass"] = (
    ~rice_analysis["rice_measurement_flag"]
    & ~rice_analysis["rice_extreme_yield_flag"]
)

print("\nProvisional screening outcome:")
display(
    rice_analysis["rice_strict_screen_pass"]
    .value_counts(dropna=False)
    .rename_axis("strict_screen_pass")
    .reset_index(name="row_count")
)

print("\nScreening limitations:")
print(
    "The threshold of 10000 is a provisional investigation threshold, "
    "not a verified agronomic validity limit."
)
print(
    "Source units and yield derivation still need verification. "
    "Weather alignment remains a starting-calendar-year proxy."
)

,check,rows,pct_of_rice_subset
0,All selected Rice records,17082,100.00
1,Yield above 5000,165,0.97
2,Yield above 10000,5,0.03
3,Zero yield,13,0.08
4,Zero area and positive production,379,2.22
5,Positive area and zero production,57,0.33
6,Positive yield without positive area and produ...,1633,9.56
7,Long district-to-weather-grid distance,7506,43.94



Provisional screening outcome:


,strict_screen_pass,row_count
0,True,15444
1,False,1638



Screening limitations:
The threshold of 10000 is a provisional investigation threshold, not a verified agronomic validity limit.
Source units and yield derivation still need verification. Weather alignment remains a starting-calendar-year proxy.


In [35]:
# Create a provisional strict Rice subset and an exclusion audit.
# The original rice_analysis DataFrame remains unchanged in row coverage.

rice_strict = rice_analysis.loc[
    rice_analysis["rice_strict_screen_pass"]
].copy()

rice_exclusion_audit = rice_analysis.loc[
    ~rice_analysis["rice_strict_screen_pass"]
].copy()

# Record explicit exclusion reasons so each removed row can be traced.
rice_exclusion_audit["exclusion_reason"] = rice_exclusion_audit.apply(
    lambda row: "; ".join(
        reason for reason, condition in [
            (
                "Area zero with positive production",
                row["QC_AreaZero-ProductionPositive"] == True
            ),
            (
                "Area positive with zero production",
                row["QC_AreaPositive-ProductionZero"] == True
            ),
            (
                "Positive yield without positive area and production",
                row["QC_PositiveYieldWithoutPositiveAreaProduction"] == True
            ),
            (
                "Yield above provisional threshold of 10000",
                row["Yield"] > 10000
            )
        ]
        if condition
    ),
    axis=1
)

# Summarize the subset sizes and retained coverage.
rice_subset_summary = pd.DataFrame([
    {
        "dataset": "Rice analysis before strict screening",
        "rows": len(rice_analysis),
        "districts": rice_analysis["apy_geography_id"].nunique(),
        "states": rice_analysis["State"].nunique()
    },
    {
        "dataset": "Rice strict provisional subset",
        "rows": len(rice_strict),
        "districts": rice_strict["apy_geography_id"].nunique(),
        "states": rice_strict["State"].nunique()
    },
    {
        "dataset": "Rice exclusion audit",
        "rows": len(rice_exclusion_audit),
        "districts": rice_exclusion_audit["apy_geography_id"].nunique(),
        "states": rice_exclusion_audit["State"].nunique()
    }
])

rice_subset_summary["pct_of_original_rows"] = (
    100 * rice_subset_summary["rows"] / len(rice_analysis)
).round(2)

display(rice_subset_summary)

print("\nExclusion reasons:")
display(
    rice_exclusion_audit["exclusion_reason"]
    .value_counts()
    .rename_axis("exclusion_reason")
    .reset_index(name="row_count")
)

,dataset,rows,districts,states,pct_of_original_rows
0,Rice analysis before strict screening,17082,614,31,100.00
1,Rice strict provisional subset,15444,588,30,90.41
2,Rice exclusion audit,1638,283,28,9.59



Exclusion reasons:


,exclusion_reason,row_count
0,Positive yield without positive area and produ...,1197
1,Area zero with positive production; Positive y...,379
2,Area positive with zero production; Positive y...,57
3,Yield above provisional threshold of 10000,5


In [36]:
# Compare Rice yield distributions before and after provisional screening.
# This is a sensitivity check, not proof that the strict subset is unbiased.

rice_distribution_comparison = pd.DataFrame([
    {
        "dataset": "Before screening",
        "rows": len(rice_analysis),
        "yield_mean": rice_analysis["Yield"].mean(),
        "yield_median": rice_analysis["Yield"].median(),
        "yield_p95": rice_analysis["Yield"].quantile(0.95),
        "yield_p99": rice_analysis["Yield"].quantile(0.99),
        "yield_max": rice_analysis["Yield"].max(),
        "long_distance_rows": int(
            rice_analysis["QC_Long_District_Grid_Distance"].eq(True).sum()
        ),
        "long_distance_pct": 100 * (
            rice_analysis["QC_Long_District_Grid_Distance"].eq(True).mean()
        )
    },
    {
        "dataset": "After provisional screening",
        "rows": len(rice_strict),
        "yield_mean": rice_strict["Yield"].mean(),
        "yield_median": rice_strict["Yield"].median(),
        "yield_p95": rice_strict["Yield"].quantile(0.95),
        "yield_p99": rice_strict["Yield"].quantile(0.99),
        "yield_max": rice_strict["Yield"].max(),
        "long_distance_rows": int(
            rice_strict["QC_Long_District_Grid_Distance"].eq(True).sum()
        ),
        "long_distance_pct": 100 * (
            rice_strict["QC_Long_District_Grid_Distance"].eq(True).mean()
        )
    }
])

numeric_cols = [
    "yield_mean", "yield_median", "yield_p95",
    "yield_p99", "yield_max", "long_distance_pct"
]
rice_distribution_comparison[numeric_cols] = (
    rice_distribution_comparison[numeric_cols].round(2)
)

display(rice_distribution_comparison)

# Check whether the retained dataset still covers the full APY year range.
rice_year_coverage = (
    rice_strict.groupby("Year")
    .agg(
        rows=("Yield", "size"),
        districts=("apy_geography_id", "nunique"),
        states=("State", "nunique"),
        median_yield=("Yield", "median"),
        mean_yield=("Yield", "mean")
    )
    .reset_index()
)

rice_year_coverage[["median_yield", "mean_yield"]] = (
    rice_year_coverage[["median_yield", "mean_yield"]].round(2)
)

display(rice_year_coverage)

,dataset,rows,yield_mean,yield_median,yield_p95,yield_p99,yield_max,long_distance_rows,long_distance_pct
0,Before screening,17082,2527.29,2479.0,4246.9,4996.19,44815.0,7506,43.94
1,After provisional screening,15444,2540.55,2492.0,4249.0,4959.00,8362.0,6786,43.94


,Year,rows,districts,states,median_yield,mean_yield
0,2013-14,1149,471,29,2202.0,2293.67
1,2014-15,1208,501,30,2201.0,2317.83
2,2015-16,1213,505,30,2098.0,2168.68
3,2016-17,1257,516,30,2407.0,2449.53
4,2017-18,1307,526,30,2407.0,2516.92
5,2018-19,1296,525,30,2453.0,2466.96
6,2019-20,1311,528,30,2576.0,2621.97
7,2020-21,1358,545,29,2676.5,2685.21
8,2021-22,1315,526,28,2565.0,2616.40
9,2022-23,1392,548,29,2781.0,2812.88


In [37]:
# Inspect the weather variables available in the strict Rice subset.
# Summarize their distributions and identify constant or highly incomplete
# columns before selecting predictors for exploratory analysis.

rice_weather_cols = [
    col for col in rice_strict.columns
    if col in weather_df.columns
    and col not in {
        "apy_geography_id",
        "calendar_year",
        "weather_proxy_calendar_year"
    }
]

print("Weather columns available for analysis:")
print(rice_weather_cols)

rice_weather_summary = pd.DataFrame([
    {
        "variable": col,
        "dtype": str(rice_strict[col].dtype),
        "non_missing": int(rice_strict[col].notna().sum()),
        "missing": int(rice_strict[col].isna().sum()),
        "unique_values": int(rice_strict[col].nunique(dropna=True)),
        "mean": rice_strict[col].mean()
            if pd.api.types.is_numeric_dtype(rice_strict[col]) else None,
        "std": rice_strict[col].std()
            if pd.api.types.is_numeric_dtype(rice_strict[col]) else None,
        "min": rice_strict[col].min()
            if pd.api.types.is_numeric_dtype(rice_strict[col]) else None,
        "median": rice_strict[col].median()
            if pd.api.types.is_numeric_dtype(rice_strict[col]) else None,
        "max": rice_strict[col].max()
            if pd.api.types.is_numeric_dtype(rice_strict[col]) else None
    }
    for col in rice_weather_cols
])

display(rice_weather_summary)

Weather columns available for analysis:
['tmin_mean_c', 'tmin_min_c', 'tmax_mean_c', 'tmax_max_c', 'precipitation_total_mm', 'precipitation_mean_mm_day', 'precipitation_max_mm_day', 'tmin_valid_days', 'tmax_valid_days', 'precipitation_valid_days', 'QC_Extreme_Annual_Min_Temperature', 'QC_Extreme_Annual_Max_Temperature', 'QC_High_Annual_Precipitation', 'QC_Extreme_Daily_Precipitation', 'QC_Long_District_Grid_Distance']


,variable,dtype,non_missing,missing,unique_values,mean,std,min,median,max
0,tmin_mean_c,float64,15444,0,5077,19.274979,3.884163,-7.180329,19.854466,27.557404
1,tmin_min_c,float64,15444,0,1841,6.324531,6.104507,-30.480000,5.670000,24.210000
2,tmax_mean_c,float64,15444,0,5079,30.504619,4.239853,3.371753,31.635507,36.641671
3,tmax_max_c,float64,15444,0,1710,41.909521,5.771326,14.190000,43.580000,49.690000
4,precipitation_total_mm,float64,15444,0,5050,1389.749489,706.971565,226.390000,1220.865000,6693.070000
5,precipitation_mean_mm_day,float64,15444,0,5075,3.804746,1.935045,0.620247,3.343388,18.337178
6,precipitation_max_mm_day,float64,15444,0,4026,74.974299,37.064464,13.340000,67.430000,404.510000
7,tmin_valid_days,float64,15444,0,2,365.255051,0.435904,365.000000,365.000000,366.000000
8,tmax_valid_days,float64,15444,0,2,365.255051,0.435904,365.000000,365.000000,366.000000
9,precipitation_valid_days,float64,15444,0,2,365.255051,0.435904,365.000000,365.000000,366.000000


In [38]:
# Inspect correlations among continuous weather variables.
# High correlations can indicate overlapping information, but do not by
# themselves prove that a variable should be removed.

rice_weather_numeric_cols = [
    "tmin_mean_c",
    "tmin_min_c",
    "tmax_mean_c",
    "tmax_max_c",
    "precipitation_total_mm",
    "precipitation_mean_mm_day",
    "precipitation_max_mm_day",
]

rice_weather_correlation = rice_strict[
    rice_weather_numeric_cols + ["Yield"]
].corr(numeric_only=True)

print("Weather variables and Rice yield: correlation matrix")
display(rice_weather_correlation.round(3))

# Inspect the stored values and counts for each weather quality flag.
rice_weather_flag_cols = [
    "QC_Extreme_Annual_Min_Temperature",
    "QC_Extreme_Annual_Max_Temperature",
    "QC_High_Annual_Precipitation",
    "QC_Extreme_Daily_Precipitation",
    "QC_Long_District_Grid_Distance",
]

rice_weather_flag_summary = pd.DataFrame([
    {
        "flag": col,
        "values_and_counts": rice_strict[col]
            .value_counts(dropna=False)
            .to_dict()
    }
    for col in rice_weather_flag_cols
])

display(rice_weather_flag_summary)

Weather variables and Rice yield: correlation matrix


,tmin_mean_c,tmin_min_c,tmax_mean_c,tmax_max_c,precipitation_total_mm,precipitation_mean_mm_day,precipitation_max_mm_day,Yield
tmin_mean_c,1.000,0.851,0.874,0.592,0.006,0.006,0.062,0.294
tmin_min_c,0.851,1.000,0.590,0.189,0.152,0.152,0.069,0.307
tmax_mean_c,0.874,0.590,1.000,0.860,-0.291,-0.291,-0.050,0.274
tmax_max_c,0.592,0.189,0.860,1.000,-0.436,-0.437,-0.047,0.185
precipitation_total_mm,0.006,0.152,-0.291,-0.436,1.000,1.000,0.511,-0.176
precipitation_mean_mm_day,0.006,0.152,-0.291,-0.437,1.000,1.000,0.511,-0.176
precipitation_max_mm_day,0.062,0.069,-0.050,-0.047,0.511,0.511,1.000,-0.107
Yield,0.294,0.307,0.274,0.185,-0.176,-0.176,-0.107,1.000


,flag,values_and_counts
0,QC_Extreme_Annual_Min_Temperature,"{False: 15442, True: 2}"
1,QC_Extreme_Annual_Max_Temperature,"{False: 14518, True: 926}"
2,QC_High_Annual_Precipitation,"{False: 15414, True: 30}"
3,QC_Extreme_Daily_Precipitation,"{False: 15425, True: 19}"
4,QC_Long_District_Grid_Distance,"{False: 8658, True: 6786}"


In [39]:
# Inspect records flagged for extreme annual maximum temperature.
# Keep the flagged observations for review rather than deleting them
# without checking the source and QC definition.

rice_extreme_temp = rice_strict.loc[
    rice_strict["QC_Extreme_Annual_Max_Temperature"].eq(True)
].copy()

print("Rows flagged for extreme annual maximum temperature:",
      len(rice_extreme_temp))

display(
    rice_extreme_temp[
        [
            "State", "District", "Year", "Season", "Yield",
            "tmax_mean_c", "tmax_max_c",
            "QC_Extreme_Annual_Max_Temperature",
            "QC_Long_District_Grid_Distance"
        ]
    ]
    .sort_values("tmax_max_c", ascending=False)
    .head(20)
)

# Compare flagged and unflagged records to understand the scale of the
# difference. This is descriptive, not a validation of the flag.
rice_temp_flag_comparison = (
    rice_strict.groupby("QC_Extreme_Annual_Max_Temperature", dropna=False)
    .agg(
        rows=("Yield", "size"),
        median_yield=("Yield", "median"),
        mean_yield=("Yield", "mean"),
        median_tmax_mean_c=("tmax_mean_c", "median"),
        median_tmax_max_c=("tmax_max_c", "median"),
        long_distance_pct=(
            "QC_Long_District_Grid_Distance",
            lambda s: 100 * s.eq(True).mean()
        )
    )
    .reset_index()
)

rice_temp_flag_comparison[
    [
        "median_yield", "mean_yield",
        "median_tmax_mean_c", "median_tmax_max_c",
        "long_distance_pct"
    ]
] = rice_temp_flag_comparison[
    [
        "median_yield", "mean_yield",
        "median_tmax_mean_c", "median_tmax_max_c",
        "long_distance_pct"
    ]
].round(2)

display(rice_temp_flag_comparison)

# Define a compact set of nonredundant candidate weather features.
# This is a starting feature set, not a claim that these are the optimal
# predictors for Rice yield.
rice_candidate_weather_features = [
    "tmin_mean_c",
    "tmax_mean_c",
    "precipitation_total_mm",
    "precipitation_max_mm_day"
]

print("Candidate weather features:", rice_candidate_weather_features)
print(
    "Note: annual weather summaries are matched using the starting-year "
    "proxy; crop-season alignment is not verified."
)

Rows flagged for extreme annual maximum temperature: 926


,State,District,Year,Season,Yield,tmax_mean_c,tmax_max_c,QC_Extreme_Annual_Max_Temperature,QC_Long_District_Grid_Distance
58378,Uttar Pradesh,Banda,2014-15,Total,1785.0,33.814164,49.69,True,False
58377,Uttar Pradesh,Banda,2014-15,Kharif,1785.0,33.814164,49.69,True,False
359559,Madhya Pradesh,Morena,2024-25,Kharif,2416.0,32.401585,49.62,True,True
59545,Uttar Pradesh,Jalaun,2014-15,Total,1749.0,33.667562,49.62,True,False
59027,Uttar Pradesh,Fatehpur,2014-15,Kharif,1946.0,33.942384,49.62,True,True
59029,Uttar Pradesh,Fatehpur,2014-15,Total,1946.0,33.942384,49.62,True,True
59544,Uttar Pradesh,Jalaun,2014-15,Kharif,1749.0,33.667562,49.62,True,False
359560,Madhya Pradesh,Morena,2024-25,Total,2416.0,32.401585,49.62,True,True
371379,Uttar Pradesh,Agra,2024-25,Kharif,2729.0,32.456066,49.56,True,False
371381,Uttar Pradesh,Agra,2024-25,Total,2729.0,32.456066,49.56,True,False


,QC_Extreme_Annual_Max_Temperature,rows,median_yield,mean_yield,median_tmax_mean_c,median_tmax_max_c,long_distance_pct
0,False,14518,2461.0,2520.15,31.51,43.23,44.30
1,True,926,2744.0,2860.33,32.75,48.40,38.23


Candidate weather features: ['tmin_mean_c', 'tmax_mean_c', 'precipitation_total_mm', 'precipitation_max_mm_day']
Note: annual weather summaries are matched using the starting-year proxy; crop-season alignment is not verified.


In [40]:
# Check whether Rice records share identical yield and weather values across
# seasons within the same district and agricultural year.
# Do not remove rows here: the purpose is to quantify overlap first.

rice_district_year_season_check = (
    rice_strict.groupby(
        ["apy_geography_id", "Year", "Season"],
        dropna=False
    )
    .agg(
        rows=("Yield", "size"),
        yield_value=("Yield", "first"),
        yield_unique=("Yield", "nunique"),
        tmin_mean_unique=("tmin_mean_c", "nunique"),
        tmax_mean_unique=("tmax_mean_c", "nunique"),
        rainfall_unique=("precipitation_total_mm", "nunique")
    )
    .reset_index()
)

# Count district-year combinations with both Kharif and Total records.
rice_kharif_total = rice_strict.loc[
    rice_strict["Season"].isin(["Kharif", "Total"])
].copy()

rice_kharif_total_comparison = (
    rice_kharif_total.pivot_table(
        index=["apy_geography_id", "Year"],
        columns="Season",
        values=[
            "Yield",
            "tmin_mean_c",
            "tmax_mean_c",
            "precipitation_total_mm"
        ],
        aggfunc="first"
    )
)

required_pairs = [
    ("Yield", "Kharif", "Total"),
    ("tmin_mean_c", "Kharif", "Total"),
    ("tmax_mean_c", "Kharif", "Total"),
    ("precipitation_total_mm", "Kharif", "Total")
]

comparison_rows = []

for variable, season_a, season_b in required_pairs:
    if (
        (variable, season_a) in rice_kharif_total_comparison.columns
        and (variable, season_b) in rice_kharif_total_comparison.columns
    ):
        a = rice_kharif_total_comparison[(variable, season_a)]
        b = rice_kharif_total_comparison[(variable, season_b)]
        paired = a.notna() & b.notna()

        comparison_rows.append({
            "variable": variable,
            "district_year_pairs": int(paired.sum()),
            "identical_pairs": int((a[paired] == b[paired]).sum()),
            "identical_pct": round(
                100 * (a[paired] == b[paired]).mean(), 2
            ) if paired.any() else None
        })

display(pd.DataFrame(comparison_rows))

print("Season labels in the strict Rice subset:")
display(
    rice_strict["Season"]
    .value_counts(dropna=False)
    .rename_axis("Season")
    .reset_index(name="row_count")
)

,variable,district_year_pairs,identical_pairs,identical_pct
0,Yield,5838,3592,61.53
1,tmin_mean_c,5838,5838,100.00
2,tmax_mean_c,5838,5838,100.00
3,precipitation_total_mm,5838,5838,100.00


Season labels in the strict Rice subset:


,Season,row_count
0,Total,6298
1,Kharif,5838
2,Rabi,2010
3,Summer,1298


In [41]:
# Quantify the number of Rice season records per district-year.
# This helps identify repeated outcomes and overlapping reporting categories
# before choosing the unit of analysis.

rice_season_count_by_district_year = (
    rice_strict.groupby(
        ["apy_geography_id", "Year"],
        dropna=False
    )
    .agg(
        season_record_count=("Season", "nunique"),
        seasons_present=("Season", lambda s: ", ".join(sorted(s.unique()))),
        rice_rows=("Yield", "size"),
        distinct_yield_values=("Yield", "nunique"),
        distinct_weather_profiles=(
            "tmax_mean_c",
            "nunique"
        )
    )
    .reset_index()
)

print("Number of district-year combinations by season count:")
display(
    rice_season_count_by_district_year["season_record_count"]
    .value_counts()
    .sort_index()
    .rename_axis("season_count")
    .reset_index(name="district_year_combinations")
)

print("\nExamples with all four season labels:")
display(
    rice_season_count_by_district_year.loc[
        rice_season_count_by_district_year["season_record_count"].eq(4)
    ].head(15)
)

# Check whether Total yield equals any seasonal yield for paired district-years.
rice_total_season_check = rice_strict.loc[
    rice_strict["Season"].isin(["Total", "Kharif", "Rabi", "Summer"])
].pivot_table(
    index=["apy_geography_id", "Year"],
    columns="Season",
    values="Yield",
    aggfunc="first"
)

total_comparison_rows = []

if "Total" in rice_total_season_check.columns:
    for season in ["Kharif", "Rabi", "Summer"]:
        if season in rice_total_season_check.columns:
            total_yield = rice_total_season_check["Total"]
            season_yield = rice_total_season_check[season]
            paired = total_yield.notna() & season_yield.notna()

            total_comparison_rows.append({
                "comparison": f"Total vs {season}",
                "paired_district_years": int(paired.sum()),
                "identical_yield_pairs": int(
                    (total_yield[paired] == season_yield[paired]).sum()
                ),
                "identical_pct": round(
                    100 * (total_yield[paired] == season_yield[paired]).mean(),
                    2
                ) if paired.any() else None
            })

display(pd.DataFrame(total_comparison_rows))

Number of district-year combinations by season count:


,season_count,district_year_combinations
0,1,7
1,2,4171
2,3,1393
3,4,729



Examples with all four season labels:


,apy_geography_id,Year,season_record_count,seasons_present,rice_rows,distinct_yield_values,distinct_weather_profiles
434,APY-GEO-0055,2013-14,4,"Kharif, Rabi, Summer, Total",4,4,1
435,APY-GEO-0055,2014-15,4,"Kharif, Rabi, Summer, Total",4,4,1
436,APY-GEO-0055,2015-16,4,"Kharif, Rabi, Summer, Total",4,4,1
437,APY-GEO-0055,2016-17,4,"Kharif, Rabi, Summer, Total",4,4,1
438,APY-GEO-0055,2017-18,4,"Kharif, Rabi, Summer, Total",4,4,1
439,APY-GEO-0055,2018-19,4,"Kharif, Rabi, Summer, Total",4,4,1
440,APY-GEO-0055,2019-20,4,"Kharif, Rabi, Summer, Total",4,4,1
441,APY-GEO-0055,2020-21,4,"Kharif, Rabi, Summer, Total",4,4,1
442,APY-GEO-0055,2021-22,4,"Kharif, Rabi, Summer, Total",4,4,1
443,APY-GEO-0055,2022-23,4,"Kharif, Rabi, Summer, Total",4,4,1


,comparison,paired_district_years,identical_yield_pairs,identical_pct
0,Total vs Kharif,5838,3592,61.53
1,Total vs Rabi,2009,193,9.61
2,Total vs Summer,1297,52,4.01


In [42]:
# Create a Kharif-only Rice dataset for the first exploratory analysis.
# Keep the source subset and excluded seasons intact.
# Weather is still an annual starting-year proxy, not verified Kharif weather.

rice_kharif = rice_strict.loc[
    rice_strict["Season"].eq("Kharif")
].copy()

# Summarize the analysis population and its geographic/year coverage.
rice_kharif_summary = pd.DataFrame([{
    "dataset": "Kharif Rice — provisional strict subset",
    "rows": len(rice_kharif),
    "districts": rice_kharif["apy_geography_id"].nunique(),
    "states": rice_kharif["State"].nunique(),
    "agricultural_years": rice_kharif["Year"].nunique(),
    "first_year": rice_kharif["Year"].min(),
    "last_year": rice_kharif["Year"].max(),
    "median_yield": rice_kharif["Yield"].median(),
    "mean_yield": rice_kharif["Yield"].mean(),
    "long_distance_rows": int(
        rice_kharif["QC_Long_District_Grid_Distance"].eq(True).sum()
    ),
    "long_distance_pct": round(
        100 * rice_kharif["QC_Long_District_Grid_Distance"].eq(True).mean(),
        2
    )
}])

display(rice_kharif_summary)

# Verify that there is at most one Kharif row per district and APY year.
duplicate_kharif_keys = rice_kharif.duplicated(
    subset=["apy_geography_id", "Year"],
    keep=False
)

print("Rows participating in duplicate district-year keys:",
      int(duplicate_kharif_keys.sum()))

if duplicate_kharif_keys.any():
    display(
        rice_kharif.loc[
            duplicate_kharif_keys,
            ["State", "District", "Year", "Season", "Yield"]
        ].sort_values(["State", "District", "Year"]).head(20)
    )
else:
    print("Verified: one Kharif record per district-year in this subset.")

,dataset,rows,districts,states,agricultural_years,first_year,last_year,median_yield,mean_yield,long_distance_rows,long_distance_pct
0,Kharif Rice — provisional strict subset,5838,584,30,12,2013-14,2024-25,2351.0,2411.183796,2566,43.95


Rows participating in duplicate district-year keys: 0
Verified: one Kharif record per district-year in this subset.


In [43]:
# Summarize Kharif Rice yield and annual weather proxies by APY year.
# These are descriptive associations; weather uses the starting-calendar-year
# proxy, and full Kharif-season alignment has not been verified.

rice_kharif_yearly = (
    rice_kharif.groupby("Year")
    .agg(
        rows=("Yield", "size"),
        districts=("apy_geography_id", "nunique"),
        median_yield=("Yield", "median"),
        mean_yield=("Yield", "mean"),
        mean_tmin_c=("tmin_mean_c", "mean"),
        mean_tmax_c=("tmax_mean_c", "mean"),
        mean_rainfall_mm=("precipitation_total_mm", "mean"),
        mean_max_daily_rainfall_mm=("precipitation_max_mm_day", "mean"),
        long_distance_pct=(
            "QC_Long_District_Grid_Distance",
            lambda s: 100 * s.eq(True).mean()
        )
    )
    .reset_index()
)

rice_kharif_yearly[
    [
        "median_yield", "mean_yield",
        "mean_tmin_c", "mean_tmax_c",
        "mean_rainfall_mm", "mean_max_daily_rainfall_mm",
        "long_distance_pct"
    ]
] = rice_kharif_yearly[
    [
        "median_yield", "mean_yield",
        "mean_tmin_c", "mean_tmax_c",
        "mean_rainfall_mm", "mean_max_daily_rainfall_mm",
        "long_distance_pct"
    ]
].round(2)

display(rice_kharif_yearly)

# Measure raw correlations within the Kharif subset.
# Correlations do not adjust for geography, time trends, or data quality.
rice_kharif_corr_cols = [
    "Yield",
    "tmin_mean_c",
    "tmax_mean_c",
    "precipitation_total_mm",
    "precipitation_max_mm_day"
]

display(
    rice_kharif[rice_kharif_corr_cols]
    .corr(numeric_only=True)
    .round(3)
)

print(
    "Interpretation limit: these correlations are exploratory, "
    "not causal estimates. Weather is an annual starting-year proxy."
)

,Year,rows,districts,median_yield,mean_yield,mean_tmin_c,mean_tmax_c,mean_rainfall_mm,mean_max_daily_rainfall_mm,long_distance_pct
0,2013-14,437,437,2117.0,2223.16,18.66,30.11,1389.55,79.59,44.16
1,2014-15,466,466,2071.0,2221.67,18.55,30.45,1098.11,74.12,44.21
2,2015-16,466,466,2022.0,2110.03,18.67,30.42,1232.64,75.31,44.85
3,2016-17,474,474,2287.5,2293.52,19.06,30.90,1237.04,62.76,44.09
4,2017-18,478,478,2313.5,2324.83,18.83,30.66,1347.28,78.90,43.51
5,2018-19,479,479,2291.0,2330.97,18.54,30.49,1186.63,62.22,44.47
6,2019-20,477,477,2453.0,2482.27,18.56,29.96,1415.48,71.08,44.23
7,2020-21,489,489,2530.0,2552.78,18.33,29.39,1518.45,79.57,43.76
8,2021-22,480,480,2434.5,2496.46,18.97,30.09,1395.25,72.28,43.12
9,2022-23,496,496,2568.5,2638.34,18.94,30.21,1413.93,74.56,43.75


,Yield,tmin_mean_c,tmax_mean_c,precipitation_total_mm,precipitation_max_mm_day
Yield,1.000,0.255,0.260,-0.228,-0.121
tmin_mean_c,0.255,1.000,0.904,0.008,0.071
tmax_mean_c,0.260,0.904,1.000,-0.264,-0.024
precipitation_total_mm,-0.228,0.008,-0.264,1.000,0.502
precipitation_max_mm_day,-0.121,0.071,-0.024,0.502,1.000


Interpretation limit: these correlations are exploratory, not causal estimates. Weather is an annual starting-year proxy.


In [44]:
# Identify districts with a Kharif Rice observation in every agricultural year.
# This creates a balanced district panel for a fairer descriptive comparison.

expected_years = rice_kharif["Year"].nunique()

district_year_counts = (
    rice_kharif.groupby("apy_geography_id")["Year"]
    .nunique()
)

balanced_district_ids = district_year_counts[
    district_year_counts.eq(expected_years)
].index

rice_kharif_balanced = rice_kharif.loc[
    rice_kharif["apy_geography_id"].isin(balanced_district_ids)
].copy()

print(f"Agricultural years in dataset: {expected_years}")
print(f"Districts in original subset: {rice_kharif['apy_geography_id'].nunique()}")
print(f"Districts represented in every year: {len(balanced_district_ids)}")
print(f"Balanced panel rows: {len(rice_kharif_balanced):,}")

# Compare annual yield trends in the original and balanced district samples.
balanced_yearly = (
    rice_kharif_balanced.groupby("Year")
    .agg(
        districts=("apy_geography_id", "nunique"),
        median_yield=("Yield", "median"),
        mean_yield=("Yield", "mean")
    )
    .reset_index()
)

original_yearly = (
    rice_kharif.groupby("Year")
    .agg(
        original_districts=("apy_geography_id", "nunique"),
        original_median_yield=("Yield", "median"),
        original_mean_yield=("Yield", "mean")
    )
    .reset_index()
)

yield_trend_comparison = original_yearly.merge(
    balanced_yearly,
    on="Year",
    how="left",
    validate="one_to_one"
)

display(yield_trend_comparison.round(2))

# Check weather-yield correlations within the balanced district sample.
# These remain observational and use the unverified starting-year weather proxy.
display(
    rice_kharif_balanced[
        [
            "Yield",
            "tmin_mean_c",
            "tmax_mean_c",
            "precipitation_total_mm",
            "precipitation_max_mm_day"
        ]
    ]
    .corr(numeric_only=True)
    .round(3)
)

Agricultural years in dataset: 12
Districts in original subset: 584
Districts represented in every year: 368
Balanced panel rows: 4,416


,Year,original_districts,original_median_yield,original_mean_yield,districts,median_yield,mean_yield
0,2013-14,437,2117.0,2223.16,368,2202.5,2249.24
1,2014-15,466,2071.0,2221.67,368,2189.5,2262.66
2,2015-16,466,2022.0,2110.03,368,2064.5,2142.44
3,2016-17,474,2287.5,2293.52,368,2310.5,2320.79
4,2017-18,478,2313.5,2324.83,368,2387.5,2408.42
5,2018-19,479,2291.0,2330.97,368,2414.0,2389.53
6,2019-20,477,2453.0,2482.27,368,2536.0,2529.01
7,2020-21,489,2530.0,2552.78,368,2599.0,2603.39
8,2021-22,480,2434.5,2496.46,368,2460.5,2501.84
9,2022-23,496,2568.5,2638.34,368,2565.0,2616.83


,Yield,tmin_mean_c,tmax_mean_c,precipitation_total_mm,precipitation_max_mm_day
Yield,1.000,0.254,0.271,-0.247,-0.126
tmin_mean_c,0.254,1.000,0.882,0.014,0.061
tmax_mean_c,0.271,0.882,1.000,-0.292,-0.051
precipitation_total_mm,-0.247,0.014,-0.292,1.000,0.502
precipitation_max_mm_day,-0.126,0.061,-0.051,0.502,1.000


In [45]:
# Calculate within-district deviations using the balanced panel.
# Subtracting each district's long-run mean removes time-invariant
# differences between districts, but does not control for year effects.

within_vars = [
    "Yield",
    "tmin_mean_c",
    "tmax_mean_c",
    "precipitation_total_mm",
    "precipitation_max_mm_day"
]

rice_kharif_within = rice_kharif_balanced.copy()

rice_kharif_within[within_vars] = (
    rice_kharif_within[within_vars]
    - rice_kharif_within.groupby("apy_geography_id")[within_vars]
    .transform("mean")
)

print("Within-district deviations: descriptive statistics")
display(rice_kharif_within[within_vars].describe().round(3))

print("Within-district correlation matrix")
display(
    rice_kharif_within[within_vars]
    .corr(numeric_only=True)
    .round(3)
)

print(
    "Caution: these are within-district correlations, not causal estimates. "
    "Year effects are not removed, and weather remains a starting-year proxy."
)

Within-district deviations: descriptive statistics


,Yield,tmin_mean_c,tmax_mean_c,precipitation_total_mm,precipitation_max_mm_day
count,4416.000,4416.000,4416.000,4416.000,4416.000
mean,0.000,-0.000,0.000,-0.000,-0.000
std,429.111,0.355,0.648,292.759,32.173
min,-2254.500,-1.369,-1.854,-1334.327,-82.034
25%,-233.000,-0.226,-0.407,-161.901,-18.910
50%,16.458,-0.022,0.038,-17.678,-5.467
75%,221.104,0.240,0.391,145.257,12.058
max,2750.000,1.254,1.878,3123.852,313.662


Within-district correlation matrix


,Yield,tmin_mean_c,tmax_mean_c,precipitation_total_mm,precipitation_max_mm_day
Yield,1.000,-0.002,-0.155,0.074,-0.007
tmin_mean_c,-0.002,1.000,0.650,-0.140,-0.103
tmax_mean_c,-0.155,0.650,1.000,-0.535,-0.257
precipitation_total_mm,0.074,-0.140,-0.535,1.000,0.495
precipitation_max_mm_day,-0.007,-0.103,-0.257,0.495,1.000


Caution: these are within-district correlations, not causal estimates. Year effects are not removed, and weather remains a starting-year proxy.


In [49]:
# Fit a two-way fixed-effects regression using NumPy.
# The double-demeaning transformation removes district and year effects
# in this balanced panel. Coefficients are conditional associations,
# not causal effects.

model_vars = [
    "Yield",
    "tmin_mean_c",
    "tmax_mean_c",
    "precipitation_total_mm",
    "precipitation_max_mm_day"
]

# Keep a separate working copy and ensure a consistent row order.
twfe_data = rice_kharif_balanced[
    ["apy_geography_id", "Year"] + model_vars
].copy()

# Double-demean each model variable:
# value - district mean - year mean + overall mean.
# This is valid here because the panel is balanced.
twfe_within = twfe_data[model_vars].copy()

for column in model_vars:
    twfe_within[column] = (
        twfe_data[column]
        - twfe_data.groupby("apy_geography_id")[column].transform("mean")
        - twfe_data.groupby("Year")[column].transform("mean")
        + twfe_data[column].mean()
    )

# Estimate coefficients with least squares.
weather_features = [
    "tmin_mean_c",
    "tmax_mean_c",
    "precipitation_total_mm",
    "precipitation_max_mm_day"
]

X = twfe_within[weather_features].to_numpy(dtype=float)
y = twfe_within["Yield"].to_numpy(dtype=float)

coefficients, residuals, rank, singular_values = np.linalg.lstsq(
    X, y, rcond=None
)

twfe_coefficient_results = pd.DataFrame({
    "weather_variable": weather_features,
    "coefficient": coefficients
})

display(twfe_coefficient_results.round(4))

print("Design matrix rank:", rank, "of", X.shape[1])
print("Observations:", len(y))
print(
    "Reminder: weather is an unverified starting-calendar-year proxy; "
    "coefficients are not causal effects."
)

,weather_variable,coefficient
0,tmin_mean_c,53.3590
1,tmax_mean_c,-96.6838
2,precipitation_total_mm,-0.0911
3,precipitation_max_mm_day,-0.3389


Design matrix rank: 4 of 4
Observations: 4416
Reminder: weather is an unverified starting-calendar-year proxy; coefficients are not causal effects.


In [50]:
# Diagnose multicollinearity after removing district and year fixed effects.
# High correlations or a large condition number can indicate unstable
# individual coefficients, even when the design matrix has full rank.

# Pairwise correlations among the transformed weather predictors.
within_predictor_corr = twfe_within[weather_features].corr()

print("Within-transformed predictor correlations:")
display(within_predictor_corr.round(3))

# Standardize each transformed predictor so the condition number is not
# driven merely by differences in measurement units.
X_standardized = (
    X - X.mean(axis=0)
) / X.std(axis=0, ddof=0)

condition_number = np.linalg.cond(X_standardized)

print(f"Standardized design-matrix condition number: {condition_number:.2f}")

if condition_number < 10:
    print("Diagnostic: no strong multicollinearity signal from this condition-number check.")
elif condition_number < 30:
    print("Diagnostic: moderate multicollinearity may affect coefficient stability.")
else:
    print("Diagnostic: substantial multicollinearity may affect coefficient stability.")

Within-transformed predictor correlations:


,tmin_mean_c,tmax_mean_c,precipitation_total_mm,precipitation_max_mm_day
tmin_mean_c,1.000,0.566,-0.136,-0.114
tmax_mean_c,0.566,1.000,-0.526,-0.275
precipitation_total_mm,-0.136,-0.526,1.000,0.502
precipitation_max_mm_day,-0.114,-0.275,0.502,1.000


Standardized design-matrix condition number: 2.75
Diagnostic: no strong multicollinearity signal from this condition-number check.


In [52]:
# Calculate district-clustered standard errors for the fitted two-way
# fixed-effects model. Reuse the existing X, y, and coefficients.

from scipy import stats

# Validate that the regression inputs and balanced panel are aligned.
assert len(rice_kharif_balanced) == len(y) == X.shape[0]
assert X.shape[1] == len(coefficients) == len(weather_features)

district_ids = rice_kharif_balanced["apy_geography_id"].to_numpy()
n_obs, n_features = X.shape
n_clusters = pd.Series(district_ids).nunique()

assert n_clusters > 1, "At least two district clusters are required."

# Compute regression residuals and the sandwich covariance components.
residuals_twfe = y - X @ coefficients
bread = np.linalg.inv(X.T @ X)
meat = np.zeros((n_features, n_features))

for district_id in pd.unique(district_ids):
    cluster_mask = district_ids == district_id
    cluster_score = X[cluster_mask].T @ residuals_twfe[cluster_mask]
    meat += np.outer(cluster_score, cluster_score)

# Apply a finite-sample correction to the cluster covariance estimator.
correction = (
    n_clusters / (n_clusters - 1)
    * (n_obs - 1) / (n_obs - n_features)
)

cluster_covariance = correction * bread @ meat @ bread
cluster_se = np.sqrt(np.diag(cluster_covariance))

# Approximate confidence intervals and p-values using a t distribution.
degrees_freedom = n_clusters - 1
t_statistics = coefficients / cluster_se
p_values = 2 * stats.t.sf(np.abs(t_statistics), df=degrees_freedom)
critical_t = stats.t.ppf(0.975, df=degrees_freedom)

clustered_results = pd.DataFrame({
    "weather_variable": weather_features,
    "coefficient": coefficients,
    "clustered_std_error": cluster_se,
    "t_statistic": t_statistics,
    "p_value_approx": p_values,
    "ci_lower_95": coefficients - critical_t * cluster_se,
    "ci_upper_95": coefficients + critical_t * cluster_se
})

display(clustered_results.round(4))

print(f"Observations: {n_obs:,}")
print(f"District clusters: {n_clusters}")
print(f"Reference degrees of freedom: {degrees_freedom}")

,weather_variable,coefficient,clustered_std_error,t_statistic,p_value_approx,ci_lower_95,ci_upper_95
0,tmin_mean_c,53.3590,33.9557,1.5714,0.1169,-13.4131,120.1312
1,tmax_mean_c,-96.6838,21.5842,-4.4794,0.0000,-139.1280,-54.2396
2,precipitation_total_mm,-0.0911,0.0320,-2.8458,0.0047,-0.1541,-0.0282
3,precipitation_max_mm_day,-0.3389,0.2735,-1.2392,0.2161,-0.8768,0.1989


Observations: 4,416
District clusters: 368
Reference degrees of freedom: 367


In [53]:
# Correct the finite-sample adjustment to account for all estimated parameters:
# four weather coefficients, district effects, and year effects.

n_districts = rice_kharif_balanced["apy_geography_id"].nunique()
n_years = rice_kharif_balanced["Year"].nunique()

# Parameter count for an intercept, reference-coded district/year effects,
# and the four weather predictors.
n_total_parameters = (
    1 + (n_districts - 1) + (n_years - 1) + n_features
)

assert n_obs > n_total_parameters, "Insufficient observations for this correction."

corrected_factor = (
    n_districts / (n_districts - 1)
    * (n_obs - 1) / (n_obs - n_total_parameters)
)

corrected_covariance = corrected_factor * bread @ meat @ bread
corrected_se = np.sqrt(np.diag(corrected_covariance))

corrected_t = coefficients / corrected_se
corrected_p = 2 * stats.t.sf(
    np.abs(corrected_t),
    df=n_districts - 1
)
corrected_critical_t = stats.t.ppf(
    0.975,
    df=n_districts - 1
)

corrected_results = pd.DataFrame({
    "weather_variable": weather_features,
    "coefficient": coefficients,
    "clustered_std_error": corrected_se,
    "p_value_approx": corrected_p,
    "ci_lower_95": coefficients - corrected_critical_t * corrected_se,
    "ci_upper_95": coefficients + corrected_critical_t * corrected_se
})

display(corrected_results.round(4))

print("Total parameters counted in correction:", n_total_parameters)
print("District clusters:", n_districts)

,weather_variable,coefficient,clustered_std_error,p_value_approx,ci_lower_95,ci_upper_95
0,tmin_mean_c,53.3590,35.5154,0.1338,-16.4802,123.1982
1,tmax_mean_c,-96.6838,22.5756,0.0000,-141.0776,-52.2900
2,precipitation_total_mm,-0.0911,0.0335,0.0068,-0.1570,-0.0253
3,precipitation_max_mm_day,-0.3389,0.2861,0.2369,-0.9015,0.2236


Total parameters counted in correction: 383
District clusters: 368


In [54]:
# Save the corrected exploratory fixed-effects results and their limitations.
# These results are conditional associations, not causal estimates.

results_dir = project_root / "data" / "processed" / "weather_regional"
results_dir.mkdir(parents=True, exist_ok=True)

# Store the latest corrected coefficient estimates and uncertainty measures.
model_results_path = results_dir / "rice_kharif_twfe_clustered_results.csv"
corrected_results.to_csv(model_results_path, index=False)

# Record the sample, specification, and limitations alongside the results.
model_notes_path = results_dir / "rice_kharif_twfe_model_notes.txt"

model_notes = f"""
BharatCrop Intelligence — Rice Kharif Exploratory Fixed-Effects Model

Sample
- Observations: {n_obs}
- Districts: {n_districts}
- Agricultural years: {n_years}
- Panel: balanced district-year panel
- Crop and season: Rice, Kharif
- Yield screening: provisional strict screening applied

Specification
- Outcome: Yield
- Predictors: mean minimum temperature, mean maximum temperature,
  total precipitation, maximum daily precipitation
- District fixed effects: included
- Year fixed effects: included
- Standard errors: district-clustered sandwich estimator
- Inference: approximate; finite-sample correction and degrees-of-freedom
  conventions may affect reported uncertainty

Critical limitations
- Weather uses the starting-calendar-year proxy.
- Full agricultural-year and Kharif-season weather alignment is unverified.
- Coefficients are conditional associations, not causal effects.
- Remaining time-varying confounding and measurement error are possible.
- Geographic weather-location confidence remains a concern for some records.
- This model is exploratory and should not be presented as validated
  evidence that weather causes changes in rice yield.

Interpretation
- Retain coefficient estimates and uncertainty together.
- Do not interpret statistical significance as proof of causation.
- Revisit the model after temporal alignment and source definitions are verified.
"""

model_notes_path.write_text(model_notes.strip(), encoding="utf-8")

print("Saved model results:", model_results_path)
print("Saved model notes:", model_notes_path)
print("Results rows:", len(corrected_results))
print("Files saved successfully.")

Saved model results: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\weather_regional\rice_kharif_twfe_clustered_results.csv
Saved model notes: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\weather_regional\rice_kharif_twfe_model_notes.txt
Results rows: 4
Files saved successfully.


In [55]:
# Verify that the saved model artifacts match the current notebook results.
# This is a read-only check; it does not modify either file.

saved_results = pd.read_csv(model_results_path)
saved_notes = model_notes_path.read_text(encoding="utf-8")

assert len(saved_results) == len(corrected_results) == 4
assert saved_results["weather_variable"].tolist() == (
    corrected_results["weather_variable"].tolist()
)

np.testing.assert_allclose(
    saved_results["coefficient"].to_numpy(),
    corrected_results["coefficient"].to_numpy(),
    rtol=1e-10,
    atol=1e-10
)

required_note_phrases = [
    "starting-calendar-year proxy",
    "unverified",
    "not causal effects",
    "exploratory"
]

missing_phrases = [
    phrase for phrase in required_note_phrases
    if phrase.lower() not in saved_notes.lower()
]

print("Saved CSV matches in-memory coefficients: PASS")
print("Saved result rows:", len(saved_results))
print("Missing documentation phrases:", missing_phrases)

assert not missing_phrases, (
    f"Model notes need revision; missing phrases: {missing_phrases}"
)

print("Artifact verification: PASS")

Saved CSV matches in-memory coefficients: PASS
Saved result rows: 4
Missing documentation phrases: []
Artifact verification: PASS


In [56]:
# Inspect existing source-registry metadata for APY and weather sources.
# This checks project documentation before we assume any new definitions.

source_registry_path = project_root / "metadata" / "source_registry.csv"

source_registry = pd.read_csv(source_registry_path)

display(
    source_registry.loc[
        source_registry.astype(str)
        .apply(
            lambda column: column.str.contains(
                "APY|District-level Area|primary-estimate-report|IMD|rainfall",
                case=False,
                regex=True,
                na=False
            )
        )
        .any(axis=1)
    ].to_string(index=False)
)

'source_id                              dataset_name                                  organization                                                  official_url        category                                    access_method  years_available  geographic_coverage  schema_verified  download_verified  coverage_verified  license_verified  integration_verified     status                                                          limitations\n   SRC001 District-level Area, Production and Yield Department of Agriculture and Farmers Welfare               https://www.upag.gov.in/primary-estimate-report Crop statistics                 Portal/API; confirm requirements              NaN                  NaN            False              False              False             False                 False DISCOVERED                      Actual export and data coverage not yet audited\n   SRC003                IMD Daily Gridded Rainfall               India Meteorological Department https://www.imdpune.gov

In [57]:
# Inspect APY reporting periods and season labels.
# Observed labels are evidence of the dataset's structure,
# not proof of the official reporting-period definitions.

print("APY year labels:")
print(
    apy_df["Year"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

print("\nAPY season labels:")
print(
    apy_df["Season"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nYear-season combinations:")
print(
    apy_df.groupby(["Year", "Season"], dropna=False)
    .size()
    .rename("row_count")
    .reset_index()
    .sort_values(["Year", "Season"])
    .to_string(index=False)
)

APY year labels:
Year
2013-14    31356
2014-15    31356
2015-16    31356
2016-17    31356
2017-18    31356
2018-19    31356
2019-20    31356
2020-21    31356
2021-22    31356
2022-23    31356
2023-24    31356
2024-25    31356

APY season labels:
Season
Total     143028
Kharif    106692
Rabi       88704
Summer     37848

Year-season combinations:
   Year Season  row_count
2013-14 Kharif       8891
2013-14   Rabi       7392
2013-14 Summer       3154
2013-14  Total      11919
2014-15 Kharif       8891
2014-15   Rabi       7392
2014-15 Summer       3154
2014-15  Total      11919
2015-16 Kharif       8891
2015-16   Rabi       7392
2015-16 Summer       3154
2015-16  Total      11919
2016-17 Kharif       8891
2016-17   Rabi       7392
2016-17 Summer       3154
2016-17  Total      11919
2017-18 Kharif       8891
2017-18   Rabi       7392
2017-18 Summer       3154
2017-18  Total      11919
2018-19 Kharif       8891
2018-19   Rabi       7392
2018-19 Summer       3154
2018-19  Total      11919
20

In [60]:
# Validate the integrated APY dataset's district-crop-year-season granularity.
# Total is an aggregate category, not an individual growing season.

integration_key = [
    "apy_geography_id",
    "Crop",
    "Year",
    "Season",
]

# Count records for each intended analytical key.
key_counts = (
    apy_weather_integrated
    .groupby(integration_key, dropna=False)
    .size()
    .rename("records_per_key")
    .reset_index()
)

duplicate_keys = key_counts.loc[
    key_counts["records_per_key"] > 1
]

print("Integrated rows:", len(apy_weather_integrated))
print("Unique district-crop-year-season keys:", len(key_counts))
print("Keys with multiple records:", len(duplicate_keys))

if not duplicate_keys.empty:
    print("\nExamples of repeated keys:")
    display(
        duplicate_keys
        .sort_values("records_per_key", ascending=False)
        .head(15)
    )

# Inspect crop diversity across aggregate and seasonal records.
print("\nDistinct crops by season:")
display(
    apy_weather_integrated
    .groupby("Season")["Crop"]
    .nunique()
    .rename("distinct_crops")
    .reset_index()
)

# Check whether each district-crop-year group has an aggregate Total row.
group_key = ["apy_geography_id", "Crop", "Year"]

total_presence = (
    apy_weather_integrated
    .assign(is_total=apy_weather_integrated["Season"].eq("Total"))
    .groupby(group_key)["is_total"]
    .any()
)

print("\nDistrict-crop-year groups:", len(total_presence))
print("Groups with a Total row:", int(total_presence.sum()))
print("Groups without a Total row:", int((~total_presence).sum()))

Integrated rows: 376272
Unique district-crop-year-season keys: 376272
Keys with multiple records: 0

Distinct crops by season:


,Season,distinct_crops
0,Kharif,18
1,Rabi,18
2,Summer,16
3,Total,19



District-crop-year groups: 143028
Groups with a Total row: 143028
Groups without a Total row: 0


In [61]:
# Verify that weather integration preserved every APY record.
# Reconcile merge-status counts and check weather availability flags.

original_rows = len(apy_df)
integrated_rows = len(apy_weather_integrated)

print("Original APY rows:", original_rows)
print("Integrated rows:", integrated_rows)
print("Row count preserved:", original_rows == integrated_rows)

print("\nMerge status:")
display(
    apy_weather_integrated["_merge"]
    .value_counts(dropna=False)
    .rename_axis("merge_status")
    .reset_index(name="row_count")
)

print("\nWeather availability status:")
display(
    apy_weather_integrated["weather_match_status"]
    .value_counts(dropna=False)
    .rename_axis("weather_match_status")
    .reset_index(name="row_count")
)

# Check whether the explicit availability flag agrees with the merge result.
merge_matched = apy_weather_integrated["_merge"].eq("both")
weather_available = apy_weather_integrated["weather_available"].eq(True)

print(
    "\nRows where merge match and weather_available disagree:",
    int((merge_matched != weather_available).sum())
)

# Confirm that no rows were lost or added by the merge.
assert original_rows == integrated_rows, "Integration changed the APY row count."
assert (merge_matched == weather_available).all(), (
    "Merge status and weather availability disagree."
)

print("\nMerge integrity checks: PASS")

Original APY rows: 376272
Integrated rows: 376272
Row count preserved: True

Merge status:


,merge_status,row_count
0,both,320124
1,left_only,56148
2,right_only,0



Weather availability status:


,weather_match_status,row_count
0,MATCHED,320124
1,NO_WEATHER_COVERAGE,56148



Rows where merge match and weather_available disagree: 0

Merge integrity checks: PASS


In [62]:
# Validate the integrity of weather measurements after the merge.
# This checks existing values without imputing or modifying any records.

weather_columns = [
    "tmin_mean_c",
    "tmin_min_c",
    "tmax_mean_c",
    "tmax_max_c",
    "precipitation_total_mm",
    "precipitation_mean_mm_day",
    "precipitation_max_mm_day",
    "tmin_valid_days",
    "tmax_valid_days",
    "precipitation_valid_days",
]

matched_weather = apy_weather_integrated.loc[
    apy_weather_integrated["_merge"].eq("both")
].copy()

# Summarize missing values and infinite values in matched weather records.
weather_quality = pd.DataFrame({
    "missing_count": matched_weather[weather_columns].isna().sum(),
    "infinite_count": matched_weather[weather_columns]
        .select_dtypes(include="number")
        .apply(lambda column: np.isinf(column).sum())
        .reindex(weather_columns, fill_value=0),
})

print("Matched weather rows:", len(matched_weather))
print("\nWeather-field quality:")
display(weather_quality)

# Inspect observed valid-day ranges to detect impossible or suspicious coverage.
valid_day_columns = [
    "tmin_valid_days",
    "tmax_valid_days",
    "precipitation_valid_days",
]

print("\nValid-day count summaries:")
display(matched_weather[valid_day_columns].describe().T)

print("\nMatched rows with any weather field missing:",
      int(matched_weather[weather_columns].isna().any(axis=1).sum()))

Matched weather rows: 320124

Weather-field quality:


,missing_count,infinite_count
tmin_mean_c,0,0
tmin_min_c,0,0
tmax_mean_c,0,0
tmax_max_c,0,0
precipitation_total_mm,0,0
precipitation_mean_mm_day,0,0
precipitation_max_mm_day,0,0
tmin_valid_days,0,0
tmax_valid_days,0,0
precipitation_valid_days,0,0



Valid-day count summaries:


,count,mean,std,min,25%,50%,75%,max
tmin_valid_days,320124.0,365.25,0.433013,365.0,365.0,365.0,365.25,366.0
tmax_valid_days,320124.0,365.25,0.433013,365.0,365.0,365.0,365.25,366.0
precipitation_valid_days,320124.0,365.25,0.433013,365.0,365.0,365.0,365.25,366.0



Matched rows with any weather field missing: 0


In [63]:
# Check physical plausibility and internal consistency of matched weather data.
# Thresholds below are broad screening rules, not official IMD validation limits.

weather = matched_weather

# Temperature ordering should hold for both annual means and annual extremes.
mean_temp_inversions = weather["tmin_mean_c"] > weather["tmax_mean_c"]
extreme_temp_inversions = weather["tmin_min_c"] > weather["tmax_max_c"]

# Identify broad physical-range violations without silently removing records.
checks = {
    "negative_annual_precipitation":
        weather["precipitation_total_mm"] < 0,
    "negative_daily_max_precipitation":
        weather["precipitation_max_mm_day"] < 0,
    "mean_min_temp_above_mean_max_temp":
        mean_temp_inversions,
    "annual_min_temp_above_annual_max_temp":
        extreme_temp_inversions,
    "mean_min_temp_below_-90C":
        weather["tmin_mean_c"] < -90,
    "mean_max_temp_above_65C":
        weather["tmax_mean_c"] > 65,
    "annual_precipitation_above_10000mm":
        weather["precipitation_total_mm"] > 10000,
}

print("Weather plausibility checks:")
for check_name, flags in checks.items():
    print(f"{check_name}: {int(flags.sum()):,} rows")

print("\nWeather measurement ranges:")
display(
    weather[
        [
            "tmin_mean_c",
            "tmin_min_c",
            "tmax_mean_c",
            "tmax_max_c",
            "precipitation_total_mm",
            "precipitation_max_mm_day",
        ]
    ].describe().T
)

Weather plausibility checks:
negative_annual_precipitation: 0 rows
negative_daily_max_precipitation: 0 rows
mean_min_temp_above_mean_max_temp: 0 rows
annual_min_temp_above_annual_max_temp: 0 rows
mean_min_temp_below_-90C: 0 rows
mean_max_temp_above_65C: 0 rows
annual_precipitation_above_10000mm: 0 rows

Weather measurement ranges:


,count,mean,std,min,25%,50%,75%,max
tmin_mean_c,320124.0,19.239109,3.794166,-11.804208,18.827205,19.771452,20.622274,28.327978
tmin_min_c,320124.0,6.073443,5.936465,-35.460000,3.340000,5.380000,8.910000,26.440000
tmax_mean_c,320124.0,30.864958,4.247574,-0.886831,30.387589,31.996475,33.052732,36.641671
tmax_max_c,320124.0,42.556069,5.618969,14.190000,40.490000,44.270000,46.350000,49.840000
precipitation_total_mm,320124.0,1262.591146,645.650421,136.260000,851.730000,1124.290000,1477.020000,6693.070000
precipitation_max_mm_day,320124.0,72.452898,36.255227,13.340000,47.290000,64.700000,87.660000,404.510000


In [64]:
# Validate the APY-year to weather-calendar-year proxy mapping.
# This verifies the implemented mapping, not the agronomic validity of the proxy.

proxy_check = apy_weather_integrated[
    ["Year", "start_year", "weather_proxy_calendar_year",
     "weather_alignment_method", "weather_full_season_alignment_verified"]
].drop_duplicates().sort_values(["Year", "start_year"])

print("Distinct year-to-weather proxy combinations:", len(proxy_check))
display(proxy_check.to_string(index=False))

# Check whether each APY year maps to its starting calendar year.
year_start = (
    apy_weather_integrated["Year"]
    .astype(str)
    .str.extract(r"^(\d{4})-\d{2}$")[0]
)

expected_start_year = pd.to_numeric(year_start, errors="coerce")
actual_start_year = pd.to_numeric(
    apy_weather_integrated["start_year"], errors="coerce"
)
actual_proxy_year = pd.to_numeric(
    apy_weather_integrated["weather_proxy_calendar_year"],
    errors="coerce"
)

mapping_mismatch = (
    expected_start_year.isna()
    | actual_start_year.isna()
    | actual_proxy_year.isna()
    | expected_start_year.ne(actual_start_year)
    | actual_start_year.ne(actual_proxy_year)
)

print("\nRows with an invalid or inconsistent proxy mapping:",
      int(mapping_mismatch.sum()))

print(
    "Rows marked as full-season alignment verified:",
    int(
        apy_weather_integrated[
            "weather_full_season_alignment_verified"
        ].eq(True).sum()
    )
)

Distinct year-to-weather proxy combinations: 24


'   Year  start_year  weather_proxy_calendar_year     weather_alignment_method  weather_full_season_alignment_verified\n2013-14        2013                       2013.0 STARTING_CALENDAR_YEAR_PROXY                                   False\n2013-14        2013                          NaN STARTING_CALENDAR_YEAR_PROXY                                   False\n2014-15        2014                       2014.0 STARTING_CALENDAR_YEAR_PROXY                                   False\n2014-15        2014                          NaN STARTING_CALENDAR_YEAR_PROXY                                   False\n2015-16        2015                       2015.0 STARTING_CALENDAR_YEAR_PROXY                                   False\n2015-16        2015                          NaN STARTING_CALENDAR_YEAR_PROXY                                   False\n2016-17        2016                       2016.0 STARTING_CALENDAR_YEAR_PROXY                                   False\n2016-17        2016                          Na


Rows with an invalid or inconsistent proxy mapping: 56148
Rows marked as full-season alignment verified: 0


In [65]:
# Separate genuine proxy-mapping inconsistencies from unmatched weather records.
# Unmatched rows are excluded from the mapping check, not treated as mapping errors.

integrated = apy_weather_integrated.copy()

# Derive the expected starting year from the APY year label.
expected_start = pd.to_numeric(
    integrated["Year"].astype(str).str.extract(r"^(\d{4})-\d{2}$")[0],
    errors="coerce"
)

actual_start = pd.to_numeric(integrated["start_year"], errors="coerce")
actual_proxy = pd.to_numeric(
    integrated["weather_proxy_calendar_year"], errors="coerce"
)

matched = integrated["_merge"].eq("both")

# Only matched rows can have an observed weather proxy year to validate.
matched_mapping_errors = matched & (
    expected_start.isna()
    | actual_start.isna()
    | actual_proxy.isna()
    | expected_start.ne(actual_start)
    | actual_start.ne(actual_proxy)
)

print("Matched rows:", int(matched.sum()))
print("Unmatched rows:", int((~matched).sum()))
print("Matched rows with mapping errors:", int(matched_mapping_errors.sum()))

print(
    "Matched rows using starting-calendar-year proxy:",
    int(
        (
            matched
            & integrated["weather_alignment_method"].eq(
                "STARTING_CALENDAR_YEAR_PROXY"
            )
        ).sum()
    )
)

print(
    "Matched rows with verified full-season alignment:",
    int(
        (
            matched
            & integrated["weather_full_season_alignment_verified"].eq(True)
        ).sum()
    )
)

assert matched_mapping_errors.sum() == 0, (
    "Found inconsistent year-to-weather proxy mappings."
)

print("\nMatched-row proxy mapping check: PASS")
print("Full-season alignment verification: STILL OPEN")

Matched rows: 320124
Unmatched rows: 56148
Matched rows with mapping errors: 0
Matched rows using starting-calendar-year proxy: 320124
Matched rows with verified full-season alignment: 0

Matched-row proxy mapping check: PASS
Full-season alignment verification: STILL OPEN


In [66]:
# Consolidate the verified integration checks into a saved summary.
# Unresolved source verification and seasonal alignment remain explicit.

from pathlib import Path

integrated = apy_weather_integrated
matched_mask = integrated["_merge"].eq("both")

integration_summary = pd.DataFrame([
    {
        "check": "APY row preservation",
        "result": "PASS" if len(integrated) == len(apy_df) else "FAIL",
        "observed_value": f"{len(integrated)} integrated / {len(apy_df)} original rows",
        "limitation": "Row preservation does not validate source correctness",
    },
    {
        "check": "Analytical key uniqueness",
        "result": (
            "PASS"
            if not integrated.duplicated(
                ["apy_geography_id", "Crop", "Year", "Season"]
            ).any()
            else "FAIL"
        ),
        "observed_value": "District-crop-year-season key",
        "limitation": "Uniqueness does not establish measurement validity",
    },
    {
        "check": "Weather merge consistency",
        "result": (
            "PASS"
            if (
                integrated["_merge"].eq("both")
                == integrated["weather_available"].eq(True)
            ).all()
            else "FAIL"
        ),
        "observed_value": (
            f"{int(matched_mask.sum())} matched; "
            f"{int((~matched_mask).sum())} unmatched"
        ),
        "limitation": "Unmatched records have no weather coverage",
    },
    {
        "check": "Matched weather completeness",
        "result": (
            "PASS"
            if not integrated.loc[matched_mask, weather_columns].isna().any().any()
            else "FAIL"
        ),
        "observed_value": f"{int(matched_mask.sum())} matched rows",
        "limitation": "Completeness does not guarantee measurement accuracy",
    },
    {
        "check": "Starting-year proxy mapping",
        "result": (
            "PASS" if int(matched_mapping_errors.sum()) == 0 else "FAIL"
        ),
        "observed_value": "Matched rows use the APY starting calendar year",
        "limitation": "Full-season alignment is unverified",
    },
    {
        "check": "Full-season weather alignment",
        "result": "UNVERIFIED",
        "observed_value": "0 matched rows marked as verified",
        "limitation": "Crop-season-specific weather periods have not been established",
    },
    {
        "check": "Official source verification",
        "result": "OPEN",
        "observed_value": "APY and IMD registry records remain DISCOVERED",
        "limitation": "Schema, download, coverage, and integration verification remain open in the registry",
    },
])

summary_path = (
    project_root
    / "data"
    / "processed"
    / "weather_regional"
    / "crop_weather_integration_validation_summary.csv"
)

summary_path.parent.mkdir(parents=True, exist_ok=True)
integration_summary.to_csv(summary_path, index=False)

print("Validation summary:")
display(integration_summary)

print("\nSaved to:", summary_path)

Validation summary:


,check,result,observed_value,limitation
0,APY row preservation,PASS,376272 integrated / 376272 original rows,Row preservation does not validate source corr...
1,Analytical key uniqueness,PASS,District-crop-year-season key,Uniqueness does not establish measurement vali...
2,Weather merge consistency,PASS,320124 matched; 56148 unmatched,Unmatched records have no weather coverage
3,Matched weather completeness,PASS,320124 matched rows,Completeness does not guarantee measurement ac...
4,Starting-year proxy mapping,PASS,Matched rows use the APY starting calendar year,Full-season alignment is unverified
5,Full-season weather alignment,UNVERIFIED,0 matched rows marked as verified,Crop-season-specific weather periods have not ...
6,Official source verification,OPEN,APY and IMD registry records remain DISCOVERED,"Schema, download, coverage, and integration ve..."



Saved to: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\weather_regional\crop_weather_integration_validation_summary.csv


In [67]:
# Reopen the saved integration summary and verify its contents.
# This is a read-only reproducibility check.

saved_summary = pd.read_csv(summary_path)

expected_checks = {
    "APY row preservation",
    "Analytical key uniqueness",
    "Weather merge consistency",
    "Matched weather completeness",
    "Starting-year proxy mapping",
    "Full-season weather alignment",
    "Official source verification",
}

actual_checks = set(saved_summary["check"])

print("Saved summary rows:", len(saved_summary))
print("Missing expected checks:", sorted(expected_checks - actual_checks))
print("Unexpected checks:", sorted(actual_checks - expected_checks))

print("\nSaved validation statuses:")
display(
    saved_summary[["check", "result"]].to_string(index=False)
)

assert actual_checks == expected_checks, "Validation summary checks do not match."
assert len(saved_summary) == 7, "Unexpected number of validation checks."
assert saved_summary.loc[
    saved_summary["check"].eq("Full-season weather alignment"), "result"
].eq("UNVERIFIED").all()
assert saved_summary.loc[
    saved_summary["check"].eq("Official source verification"), "result"
].eq("OPEN").all()

print("\nSaved summary verification: PASS")

Saved summary rows: 7
Missing expected checks: []
Unexpected checks: []

Saved validation statuses:


'                        check     result\n         APY row preservation       PASS\n    Analytical key uniqueness       PASS\n    Weather merge consistency       PASS\n Matched weather completeness       PASS\n  Starting-year proxy mapping       PASS\nFull-season weather alignment UNVERIFIED\n Official source verification       OPEN'


Saved summary verification: PASS


# Notebook 14 — Crop–Weather Integration: Final Summary

## Objective

Integrate district-level Area, Production and Yield (APY) crop records with annual district-level weather data while preserving the original crop records, validating merge integrity, and documenting data-quality and temporal-alignment limitations.

## Completed Work

- Integrated APY crop records with available district-level weather data.
- Preserved all **376,272 original APY records** without introducing duplicate analytical keys.
- Matched **320,124 records (85.08%)** with weather data; **56,148 records (14.92%)** remain unmatched.
- Confirmed weather coverage for **640 of 751 APY district/state geography IDs (85.22%)**.
- Verified that matched weather fields contain no missing or infinite values and passed the implemented broad plausibility checks.
- Saved the integration validation summary and verified that all seven expected checks are present.

## Validation Results

| Validation check | Status |
|---|---|
| APY row preservation | PASS |
| Analytical key uniqueness | PASS |
| Weather merge consistency | PASS |
| Matched weather completeness | PASS |
| Starting-year proxy mapping | PASS |
| Full-season weather alignment | UNVERIFIED |
| Official source verification | OPEN |

## Important Limitations

1. **Weather alignment:** The integration uses the starting calendar year of the APY reporting year as a proxy. This does not establish that the weather period matches the actual crop-growing season.
2. **Source verification:** Official APY and IMD source metadata, schema, coverage, and integration verification remain open in the source registry.
3. **Unmatched districts:** Records without weather coverage remain unmatched. No synthetic weather observations have been created to fill these gaps.
4. **Data quality:** Passing structural and plausibility checks does not prove that every underlying measurement is accurate.
5. **Analytical use:** Integrated weather data must be treated as exploratory until temporal alignment and source verification are resolved.

## Output Artifacts

- `data/processed/weather_regional/apy_weather_integrated_exploratory.csv`
- `data/processed/weather_regional/crop_weather_integration_validation_summary.csv`

## Conclusion

**Exploratory integration is complete, but full validation is not.** The dataset is structurally consistent and reproducible under the checks performed. Its weather alignment and official-source verification remain unresolved, so downstream analyses must retain these limitations explicitly.

## Next Notebook

**Notebook 15 — Feature Engineering:** Develop leakage-aware features from the integrated dataset, document their definitions, and distinguish verified information from exploratory weather proxies.